# Análise exploratória da base Bronze

### Este notebook tem como finalidade a avaliação de execuções necessárias na transição da camada bronze para a camada silver do Data Lake.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[1]") \
    .config("spark.driver.memory", "3g") \
    .getOrCreate()

In [2]:
from pathlib import Path
bronze_dir = Path(r"C:\Users\fred\Documents\Estudo de dados\Projeto\credit-risk\data\bronze")

In [ ]:
from __future__ import annotations

print(f"{'Dataset':<45} {'Arquivos Parquet':>16}")
print("-" * 63)

for ds_path in sorted(bronze_dir.iterdir()):
    if ds_path.is_dir():
        parquet_count = len(list(ds_path.rglob("*.parquet")))
        print(f"{ds_path.name:<45} {parquet_count:>16}")


Dataset                                       Arquivos Parquet
---------------------------------------------------------------
train_applprev_1_0                                           1
train_applprev_1_1                                           1
train_applprev_2                                             1
train_base                                                   1
train_credit_bureau_a_1_0                                    1
train_credit_bureau_a_1_1                                    1
train_credit_bureau_a_1_2                                    1
train_credit_bureau_a_1_3                                    1
train_credit_bureau_a_2_0                                    1
train_credit_bureau_a_2_1                                    1
train_credit_bureau_a_2_10                                   1
train_credit_bureau_a_2_2                                    1
train_credit_bureau_a_2_3                                    1
train_credit_bureau_a_2_4                             

In [4]:
from __future__ import annotations

from pathlib import Path
from functools import reduce
from pyspark.sql import DataFrame, SparkSession


def load_logical_dataset(spark: SparkSession, bronze_dir: Path, prefix: str) -> DataFrame:
    """
    Lê e une todas as partições de um dataset lógico pelo prefixo.

    Parâmetros
    ----------
    spark      : SparkSession ativa
    bronze_dir : Path para data/bronze
    prefix     : prefixo do dataset lógico (ex: 'train_applprev_1')

    Retorna
    -------
    DataFrame unificado de todas as partições encontradas
    """
    partitions = sorted(bronze_dir.glob(f"{prefix}*"))

    if not partitions:
        raise ValueError(f"Nenhuma partição encontrada para o prefixo: '{prefix}'")

    print(f"Partições encontradas para '{prefix}':")
    for p in partitions:
        print(f"  {p.name}")

    dfs = [spark.read.parquet(str(p)) for p in partitions]

    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

## Dataset train_base (Verificações)

In [9]:
df_base = spark.read.parquet(str(bronze_dir / "train_base"))

In [12]:
df_base.show(5)

+-------+-------------+------+--------+------+--------------------+------------+--------------------+
|case_id|date_decision| MONTH|WEEK_NUM|target|             _run_id|_ingest_date|        _source_file|
+-------+-------------+------+--------+------+--------------------+------------+--------------------+
|      0|   2019-01-03|201901|       0|     0|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
|      1|   2019-01-03|201901|       0|     0|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
|      2|   2019-01-04|201901|       0|     0|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
|      3|   2019-01-03|201901|       0|     0|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
|      4|   2019-01-04|201901|       0|     1|20260226T010510Z_...|  2026-02-25|file:///C:/Users/...|
+-------+-------------+------+--------+------+--------------------+------------+--------------------+
only showing top 5 rows



In [13]:
df_base.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- date_decision: string (nullable = true)
 |-- MONTH: long (nullable = true)
 |-- WEEK_NUM: long (nullable = true)
 |-- target: long (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [19]:
from __future__ import annotations

from pyspark.sql import functions as F

total = df_base.count()

null_counts = df_base.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_base.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_base.columns],
    key=lambda x: x[2], reverse=True
)

print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


coluna                                          null_count   null_pct
----------------------------------------------------------------------
case_id                                                  0       0.0%
date_decision                                            0       0.0%
MONTH                                                    0       0.0%
WEEK_NUM                                                 0       0.0%
target                                                   0       0.0%
_run_id                                                  0       0.0%
_ingest_date                                             0       0.0%
_source_file                                             0       0.0%


In [20]:
# ── SENTINELAS ──────────────────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

numeric_cols = [c for c, t in df_base.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "MONTH", "WEEK_NUM", "target",
                              "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_base.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

print("\n=== SENTINELAS — Strings ===")

string_cols = [c for c, t in df_base.dtypes if t == "string"
               and c not in ("_run_id", "_source_file")]

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", ""]

for c in string_cols:
    counts = df_base.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")


=== SENTINELAS — Numéricos ===

=== SENTINELAS — Strings ===


In [21]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_base.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_base.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
case_id                                               1540952
date_decision                                             657
WEEK_NUM                                                   94
MONTH                                                      22
target                                                      2


In [22]:
# ── CHAVE PRIMÁRIA ───────────────────────────────────────────────────────────
print("=== UNICIDADE DA CHAVE — case_id ===")

total      = df_base.count()
unique_ids = df_base.select("case_id").distinct().count()
dups       = total - unique_ids

print(f"  Total de linhas : {total}")
print(f"  case_id únicos  : {unique_ids}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ case_id é chave primária única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar antes da Silver")
    df_base.groupBy("case_id").count() \
           .filter(F.col("count") > 1) \
           .orderBy(F.col("count").desc()) \
           .show(10, truncate=False)


=== UNICIDADE DA CHAVE — case_id ===
  Total de linhas : 1526659
  case_id únicos  : 1526659
  Duplicatas      : 0
  ✅ case_id é chave primária única


In [23]:
# ── DATAS — RANGE E CONSISTÊNCIA ─────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

# date_decision ainda é string no Bronze — inspecionar min/max antes do cast
df_base.select(
    F.min("date_decision").alias("date_decision_min"),
    F.max("date_decision").alias("date_decision_max"),
    F.min("MONTH").alias("MONTH_min"),
    F.max("MONTH").alias("MONTH_max"),
    F.min("WEEK_NUM").alias("WEEK_NUM_min"),
    F.max("WEEK_NUM").alias("WEEK_NUM_max"),
).show(truncate=False)


=== RANGE DE DATAS ===
+-----------------+-----------------+---------+---------+------------+------------+
|date_decision_min|date_decision_max|MONTH_min|MONTH_max|WEEK_NUM_min|WEEK_NUM_max|
+-----------------+-----------------+---------+---------+------------+------------+
|2019-01-01       |2020-10-05       |201901   |202010   |0           |91          |
+-----------------+-----------------+---------+---------+------------+------------+



In [25]:
df_base.filter(F.col("WEEK_NUM") < 0).count()


0

In [28]:
from pyspark.sql import functions as F

week_counts = df_base.groupBy("WEEK_NUM").count()

# Todas as semanas esperadas no range
all_weeks = spark.range(0, 92).toDF("WEEK_NUM")

# Semanas com gap (ausentes ou com count muito abaixo da média)
gaps = all_weeks.join(week_counts, on="WEEK_NUM", how="left") \
                .filter(F.col("count").isNull())

print("=== SEMANAS SEM REGISTROS ===")
gaps.orderBy("WEEK_NUM").show(truncate=False)




=== SEMANAS SEM REGISTROS ===
+--------+-----+
|WEEK_NUM|count|
+--------+-----+
+--------+-----+



In [24]:
# ── DISTRIBUIÇÃO DO TARGET ────────────────────────────────────────────────────
print("=== DISTRIBUIÇÃO DO TARGET ===")

df_base.groupBy("target") \
       .count() \
       .withColumn("pct", F.round(F.col("count") / total * 100, 2)) \
       .orderBy("target") \
       .show(truncate=False)


=== DISTRIBUIÇÃO DO TARGET ===
+------+-------+-----+
|target|count  |pct  |
+------+-------+-----+
|0     |1478665|96.86|
|1     |47994  |3.14 |
+------+-------+-----+



## 📋 Análise — `train_base`

### Schema e Tipos

| Coluna        | dtype Bronze | Ação Silver          | Justificativa                                        |
|---------------|--------------|----------------------|------------------------------------------------------|
| case_id       | long         | manter               | Chave primária, tipo correto                         |
| date_decision | string       | cast → DateType      | Data armazenada como string no Parquet Kaggle        |
| MONTH         | long         | cast → IntegerType   | Range 201901–202010 cabe em int                      |
| WEEK_NUM      | long         | cast → IntegerType   | Range 0–91 cabe em int                               |
| target        | long         | cast → IntegerType   | Variável binária (0/1), long é desnecessário         |
| _run_id       | string       | propagar             | Metadado Bronze, não alterar                         |
| _ingest_date  | date         | propagar             | Metadado Bronze, não alterar                         |
| _source_file  | string       | propagar             | Metadado Bronze, não alterar                         |

### Nulos
Nenhuma coluna apresenta nulo. Nenhuma ação necessária na Silver.

### Sentinelas
Nenhum sentinela numérico ou string detectado. Tabela limpa.

### Cardinalidade
- `case_id`: ~1.54M distintos vs 1.526M linhas — consistente com unicidade confirmada
- `date_decision`: 657 datas distintas em ~20 meses — distribuição esperada
- `MONTH` e `WEEK_NUM`: cardinalidade baixa e coerente com o range temporal
- `target`: 2 valores — binário confirmado ✅

### Chave Primária
`case_id` é chave primária única (0 duplicatas em 1.526.659 linhas). Nenhuma deduplicação necessária na Silver.

### Range de Datas
`date_decision` cobre 2019-01-01 a 2020-10-05 (~22 meses), sem datas impossíveis. Cast seguro.

### Cobertura Temporal
Todas as 92 semanas (WEEK_NUM 0–91) apresentam registros, sem gaps no calendário.  
Volume por semana estável, variando em torno de ~16.000 registros. Nenhuma ação necessária.

### Valores Negativos em WEEK_NUM
WEEK_NUM mínimo = 0. Nenhum valor negativo detectado. Domínio válido.

### Desbalanceamento do Target
96,86% classe `0` (adimplente) vs 3,14% classe `1` (inadimplente).  
⚠️ Nenhuma ação na Silver — tratamento adiado para etapa de modelagem (SMOTE, class_weight ou threshold tuning).

---

### Ações Silver — Casts

```python
df = df.withColumn("date_decision", F.to_date(F.col("date_decision"), "yyyy-MM-dd"))
df = df.withColumn("MONTH",         F.col("MONTH").cast("int"))
df = df.withColumn("WEEK_NUM",      F.col("WEEK_NUM").cast("int"))
df = df.withColumn("target",        F.col("target").cast("int"))



## Dataset train_applprev_1_0

In [32]:
from pathlib import Path

# Sobe dois níveis a partir do notebook (notebooks/ → raiz do projeto)
repo_root  = Path().resolve().parent
bronze_dir = repo_root / "data" / "bronze"

print(f"bronze_dir: {bronze_dir}")  # confirme o path antes de rodar

df_applprev_1 = load_logical_dataset(spark, bronze_dir, "train_applprev_1")


bronze_dir: C:\Users\fred\Documents\Estudo de dados\Projeto\credit-risk\data\bronze
Partições encontradas para 'train_applprev_1':
  train_applprev_1_0
  train_applprev_1_1


In [34]:
df_applprev_1.show(5)

+-------+--------------+------------+-----------------+------------------------+---------------------+------------+-----------------+--------------------------+--------------------+----------------------+---------------------+-------------------+-------------------------+---------------+-------------+------------+------------------+-------------+------------+--------------+-------------------------+---------------+-----------------+----------------+--------------------------+------------------------+-----------------+----------------+----------------------+--------------------+----------+--------------------+---------+----------------+---------------+-----------------+---------------------------+---------------------+-----------+----------+--------------------+------------+--------------------+
|case_id|actualdpd_943P|annuity_853A|approvaldate_319D|byoccupationinc_3656910L|cancelreason_3545846M|childnum_21L|creationdate_885D|credacc_actualbalance_314A|credacc_credlmt_575A|credacc_maxhi

In [39]:
for field in df_applprev_1.schema.fields:
    print(f"{field.name:<45} {str(field.dataType):<20} nullable={field.nullable}")


case_id                                       LongType()           nullable=True
actualdpd_943P                                DoubleType()         nullable=True
annuity_853A                                  DoubleType()         nullable=True
approvaldate_319D                             StringType()         nullable=True
byoccupationinc_3656910L                      DoubleType()         nullable=True
cancelreason_3545846M                         StringType()         nullable=True
childnum_21L                                  DoubleType()         nullable=True
creationdate_885D                             StringType()         nullable=True
credacc_actualbalance_314A                    DoubleType()         nullable=True
credacc_credlmt_575A                          DoubleType()         nullable=True
credacc_maxhisbal_375A                        DoubleType()         nullable=True
credacc_minhisbal_90A                         DoubleType()         nullable=True
credacc_status_367L         

In [40]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_applprev_1.count()

null_counts = df_applprev_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_applprev_1.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_applprev_1.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 6525979

coluna                                          null_count   null_pct
----------------------------------------------------------------------
revolvingaccount_394A                              6229229     95.45%
credacc_actualbalance_314A                         6205945      95.1%
credacc_maxhisbal_375A                             6205945      95.1%
credacc_minhisbal_90A                              6205945      95.1%
credacc_status_367L                                6205945      95.1%
credacc_transactions_402L                          6205945      95.1%
isdebitcard_527L                                   6062966     92.91%
byoccupationinc_3656910L                           4991531     76.49%
dtlastpmt_581D                                     4750384     72.79%
dtlastpmtallstes_3545839D                          4043621     61.96%
employedfrom_700D                                  3886478     59.55%
childnum_21L                                       3559424     

In [41]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

numeric_cols = [c for c, t in df_applprev_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1")]

for c in numeric_cols:
    counts = df_applprev_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", ""]

string_cols = [c for c, t in df_applprev_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_applprev_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")


=== SENTINELAS — Numéricos ===
  actualdpd_943P: {99: 3}
  annuity_853A: {999: 229, 9999: 20}
  credacc_actualbalance_314A: {99: 10}
  credacc_credlmt_575A: {99: 1, 9999: 2}
  credacc_maxhisbal_375A: {99: 8}
  credacc_minhisbal_90A: {99: 8}
  credamount_590A: {99: 1, 9999: 71, 99999: 11}
  currdebt_94A: {999: 2, 9999: 1}
  downpmt_134A: {99: 7, 999: 10, 9999: 4}
  mainoccupationinc_437A: {9999: 4}
  maxdpdtolerance_577P: {99: 288, 999: 48}
  outstandingdebt_522A: {999: 2, 9999: 1}

=== SENTINELAS — Strings ===


In [42]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_applprev_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_applprev_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
case_id                                               1285631
currdebt_94A                                           541771
outstandingdebt_522A                                   377028
credamount_590A                                        245947
credacc_actualbalance_314A                              99957
annuity_853A                                            86299
revolvingaccount_394A                                   85004
credacc_maxhisbal_375A                                  73455
credacc_minhisbal_90A                                   72418
credacc_credlmt_575A                                    50158
byoccupationinc_3656910L                                31000
mainoccupationinc_437A                                  28172
downpmt_134A                                            21983
profession_152M                     

In [43]:
# ── CHAVE COMPOSTA (case_id + num_group1) ────────────────────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===")

unique_keys = df_applprev_1.select("case_id", "num_group1").distinct().count()
dups        = total - unique_keys

print(f"  Total de linhas          : {total}")
print(f"  Chaves únicas            : {unique_keys}")
print(f"  Duplicatas               : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_applprev_1.groupBy("case_id", "num_group1").count() \
                 .filter(F.col("count") > 1) \
                 .orderBy(F.col("count").desc()) \
                 .show(10, truncate=False)

# Distribuição de num_group1 (quantas aplicações anteriores por case_id)
print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_applprev_1.groupBy("num_group1").count() \
             .orderBy("num_group1") \
             .show(25, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===
  Total de linhas          : 6525979
  Chaves únicas            : 6525979
  Duplicatas               : 0
  ✅ (case_id, num_group1) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+-------+
|num_group1|count  |
+----------+-------+
|0         |1221522|
|1         |991955 |
|2         |804797 |
|3         |653172 |
|4         |531208 |
|5         |432187 |
|6         |352659 |
|7         |288526 |
|8         |236723 |
|9         |194925 |
|10        |161093 |
|11        |133591 |
|12        |111283 |
|13        |92921  |
|14        |77866  |
|15        |65626  |
|16        |55222  |
|17        |46798  |
|18        |39844  |
|19        |34061  |
+----------+-------+



In [44]:
# ── DATAS — RANGE ────────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

date_cols = [
    "approvaldate_319D", "creationdate_885D", "dateactivated_425D",
    "dtlastpmt_581D", "dtlastpmtallstes_3545839D",
    "employedfrom_700D", "firstnonzeroinstldate_307D"
]

df_applprev_1.select([
    val
    for c in date_cols
    for val in (F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max"))
]).show(truncate=False)


=== RANGE DE DATAS ===
+---------------------+---------------------+---------------------+---------------------+----------------------+----------------------+------------------+------------------+-----------------------------+-----------------------------+---------------------+---------------------+------------------------------+------------------------------+
|approvaldate_319D_min|approvaldate_319D_max|creationdate_885D_min|creationdate_885D_max|dateactivated_425D_min|dateactivated_425D_max|dtlastpmt_581D_min|dtlastpmt_581D_max|dtlastpmtallstes_3545839D_min|dtlastpmtallstes_3545839D_max|employedfrom_700D_min|employedfrom_700D_max|firstnonzeroinstldate_307D_min|firstnonzeroinstldate_307D_max|
+---------------------+---------------------+---------------------+---------------------+----------------------+----------------------+------------------+------------------+-----------------------------+-----------------------------+---------------------+---------------------+--------------------

In [45]:
# ── PADRÃO DAS COLUNAS CODIFICADAS (*M) ──────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [
    "cancelreason_3545846M", "district_544M", "education_1138M",
    "postype_4733339M", "profession_152M", "rejectreason_755M",
    "rejectreasonclient_4145042M"
]

for c in code_cols:
    outliers = df_applprev_1.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") & F.col(c).isNotNull()
    ).select(c).distinct()
    count = outliers.count()
    if count > 0:
        print(f"  ⚠️  {c}: {count} valor(es) fora do padrão")
        outliers.show(5, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  cancelreason_3545846M: 1 valor(es) fora do padrão
+---------------------+
|cancelreason_3545846M|
+---------------------+
|a55475b1             |
+---------------------+

  ⚠️  district_544M: 3 valor(es) fora do padrão
+-------------+
|district_544M|
+-------------+
|a55475b1     |
|Q3_9_meet    |
|Q3_26_the    |
+-------------+

  ⚠️  education_1138M: 1 valor(es) fora do padrão
+---------------+
|education_1138M|
+---------------+
|a55475b1       |
+---------------+

  ⚠️  postype_4733339M: 1 valor(es) fora do padrão
+----------------+
|postype_4733339M|
+----------------+
|a55475b1        |
+----------------+

  ⚠️  profession_152M: 56 valor(es) fora do padrão
+---------------+
|profession_152M|
+---------------+
|Q1_11_the      |
|Q5_14_look     |
|Q5_12_is       |
|Q6_4_your      |
|Q7_11_similar  |
+---------------+
only showing top 5 rows

  ⚠️  rejectreason_755M: 1 valor(es) fora do padrão
+-----------------+
|rejectreason_755M|


## 📋 Análise — `train_applprev_1` (union de `_0` e `_1`)

### Schema e Tipos

| Coluna                       | dtype Bronze | Ação Silver        | Justificativa                                      |
|------------------------------|--------------|--------------------|----------------------------------------------------|
| case_id                      | long         | manter             | Parte da chave composta, tipo correto              |
| num_group1                   | long         | cast → IntegerType | Range 0–19, long desnecessário                     |
| actualdpd_943P               | double       | manter             | Métrica de atraso, double correto                  |
| annuity_853A                 | double       | manter             | Valor monetário, double correto                    |
| approvaldate_319D            | string       | cast → DateType    | Data armazenada como string                        |
| byoccupationinc_3656910L     | double       | manter             | Renda, double correto                              |
| cancelreason_3545846M        | string       | manter             | Código ofuscado, string correto                    |
| childnum_21L                 | double       | cast → IntegerType | Número de filhos não deve ser double               |
| creationdate_885D            | string       | cast → DateType    | Data armazenada como string                        |
| credacc_actualbalance_314A   | double       | manter             | Saldo, double correto                              |
| credacc_credlmt_575A         | double       | manter             | Limite de crédito, double correto                  |
| credacc_maxhisbal_375A       | double       | manter             | Saldo histórico, aceita negativos legítimos        |
| credacc_minhisbal_90A        | double       | manter             | Saldo histórico, aceita negativos legítimos        |
| credacc_status_367L          | string       | manter             | Categórico, string correto                         |
| credacc_transactions_402L    | double       | cast → IntegerType | Contagem de transações não deve ser double         |
| credamount_590A              | double       | manter             | Valor monetário, double correto                    |
| credtype_587L                | string       | manter             | Categórico, string correto                         |
| currdebt_94A                 | double       | manter             | Valor monetário, double correto                    |
| dateactivated_425D           | string       | cast → DateType    | Data armazenada como string                        |
| district_544M                | string       | manter             | Código ofuscado, string correto                    |
| downpmt_134A                 | double       | manter             | Valor monetário, double correto                    |
| dtlastpmt_581D               | string       | cast → DateType    | Data armazenada como string                        |
| dtlastpmtallstes_3545839D    | string       | cast → DateType    | Data armazenada como string                        |
| education_1138M              | string       | manter             | Código ofuscado, string correto                    |
| employedfrom_700D            | string       | cast → DateType    | Data armazenada como string                        |
| familystate_726L             | string       | manter             | Categórico, string correto                         |
| firstnonzeroinstldate_307D   | string       | cast → DateType    | Data armazenada como string                        |
| inittransactioncode_279L     | string       | manter             | Categórico, string correto                         |
| isbidproduct_390L            | boolean      | manter             | Booleano correto                                   |
| isdebitcard_527L             | boolean      | manter             | Booleano correto                                   |
| mainoccupationinc_437A       | double       | manter             | Renda, double correto                              |
| maxdpdtolerance_577P         | double       | manter             | Métrica de tolerância, double correto              |
| outstandingdebt_522A         | double       | manter             | Valor monetário, double correto                    |
| pmtnum_8L                    | double       | cast → IntegerType | Número de parcelas não deve ser double             |
| postype_4733339M             | string       | manter             | Código ofuscado, string correto                    |
| profession_152M              | string       | manter             | Código ofuscado, string correto                    |
| rejectreason_755M            | string       | manter             | Código ofuscado, string correto                    |
| rejectreasonclient_4145042M  | string       | manter             | Código ofuscado, string correto                    |
| revolvingaccount_394A        | double       | manter             | ID de conta codificado, não valor monetário        |
| status_219L                  | string       | manter             | Categórico, string correto                         |
| tenor_203L                   | double       | cast → IntegerType | Prazo em meses não deve ser double                 |
| _run_id                      | string       | propagar           | Metadado Bronze, não alterar                       |
| _ingest_date                 | date         | propagar           | Metadado Bronze, não alterar                       |
| _source_file                 | string       | propagar           | Metadado Bronze, não alterar                       |

### Nulos

| Classificação | Colunas | null_pct | Ação Silver |
|---|---|---|---|
| Estrutural | `credacc_*` (5 colunas), `revolvingaccount_394A`, `isdebitcard_527L` | 92–95% | Manter null — só existe quando há conta associada |
| Informativo | `byoccupationinc_3656910L`, `dtlastpmt_581D`, `dtlastpmtallstes_3545839D`, `employedfrom_700D`, `childnum_21L`, `dateactivated_425D`, `approvaldate_319D`, `familystate_726L` | 36–76% | Manter null — ausência tem significado de negócio |
| Baixo | `firstnonzeroinstldate_307D`, `pmtnum_8L`, `tenor_203L`, `annuity_853A`, `credamount_590A`, `credtype_587L`, `downpmt_134A`, `inittransactioncode_279L`, `credacc_credlmt_575A`, `mainoccupationinc_437A` | 1–10% | Manter null — imputação é decisão de feature engineering (Gold) |
| Residual | `actualdpd_943P`, `creationdate_885D`, `isbidproduct_390L`, `status_219L` | < 0.05% | Manter null |

⚠️ Nenhuma imputação na Silver — todos os nulos preservados.

### Sentinelas

Sentinelas numéricos detectados e a substituir por `null` na Silver:

| Coluna | Valores sentinela encontrados |
|---|---|
| `actualdpd_943P` | 99 |
| `annuity_853A` | 999, 9999 |
| `credacc_actualbalance_314A` | 99 |
| `credacc_credlmt_575A` | 99, 9999 |
| `credacc_maxhisbal_375A` | 99 |
| `credacc_minhisbal_90A` | 99 |
| `credamount_590A` | 99, 9999, 99999 |
| `currdebt_94A` | 999, 9999 |
| `downpmt_134A` | 99, 999, 9999 |
| `mainoccupationinc_437A` | 9999 |
| `maxdpdtolerance_577P` | 99, 999 |
| `outstandingdebt_522A` | 999, 9999 |

Nenhum sentinela string detectado.

### Cardinalidade
- `case_id`: ~1.28M distintos — cobertura parcial do `train_base` (1.526M), esperado para tabela de aplicações anteriores
- `currdebt_94A`, `outstandingdebt_522A`, `credamount_590A`: alta cardinalidade — valores contínuos, esperado
- `profession_152M`: 11.592 distintos — alta cardinalidade para código ofuscado, atenção para encoding na Gold
- `num_group1`: 20 valores distintos (0–19) — máximo de 20 aplicações anteriores por cliente
- Nenhuma coluna constante detectada

### Chave Composta
`(case_id, num_group1)` é chave composta única (0 duplicatas em 6.525.979 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição de `num_group1`
Tabela com até 20 linhas por `case_id`. O volume decresce progressivamente do grupo 0 (1.221.522) ao grupo 19 (34.061), indicando que a maioria dos clientes tem poucas aplicações anteriores — distribuição esperada e sem anomalias.

### Range de Datas
Todas as datas dentro de ranges plausíveis:
- `approvaldate_319D`: 2005-12-30 a 2020-10-19 ✅
- `creationdate_885D`: 2005-12-26 a 2020-10-19 ✅
- `dateactivated_425D`: 2006-01-01 a 2020-10-19 ✅
- `dtlastpmt_581D`: 2008-07-04 a 2020-10-19 ✅
- `dtlastpmtallstes_3545839D`: 2008-07-04 a 2020-10-19 ✅
- `employedfrom_700D`: 1961-09-15 a 2020-07-15 ✅ (datas antigas são legítimas — data de início de emprego)
- `firstnonzeroinstldate_307D`: 2006-01-26 a 2020-11-19 ✅

Casts seguros para todas as colunas de data.

### Colunas Codificadas (`*M`) — Valores Fora do Padrão
⚠️ Valores fora do padrão `P\d+_\d+_\d+` detectados em múltiplas colunas:

| Coluna | Ocorrências | Valores |
|---|---|---|
| `cancelreason_3545846M` | 1 | `a55475b1` |
| `district_544M` | 3 | `a55475b1`, `Q3_9_meet`, `Q3_26_the` |
| `education_1138M` | 1 | `a55475b1` |
| `postype_4733339M` | 1 | `a55475b1` |
| `profession_152M` | 56 | padrão `Q\d+_\d+_word` |
| `rejectreason_755M` | 1 | `a55475b1` |
| `rejectreasonclient_4145042M` | 1 | `a55475b1` |

`a55475b1` aparece em 6 colunas — provável valor sentinela do Kaggle para dado ausente/mascarado. Substituir por `null` na Silver.  
Valores `Q\d+_\d+_word` em `district_544M` e `profession_152M` seguem padrão alternativo — manter como string, documentar como anomalia no `checks_silver.py`.

---

### Ações Silver — Casts

```python
date_cols = [
    "approvaldate_319D", "creationdate_885D", "dateactivated_425D",
    "dtlastpmt_581D", "dtlastpmtallstes_3545839D",
    "employedfrom_700D", "firstnonzeroinstldate_307D"
]
for c in date_cols:
    df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))

df = df.withColumn("num_group1",            F.col("num_group1").cast("int"))
df = df.withColumn("childnum_21L",          F.col("childnum_21L").cast("int"))
df = df.withColumn("credacc_transactions_402L", F.col("credacc_transactions_402L").cast("int"))
df = df.withColumn("pmtnum_8L",             F.col("pmtnum_8L").cast("int"))
df = df.withColumn("tenor_203L",            F.col("tenor_203L").cast("int"))
```

### Ações Silver — Sentinelas → null

```python
sentinel_map = {
    "actualdpd_943P":             ,
    "annuity_853A":               ,
    "credacc_actualbalance_314A": ,
    "credacc_credlmt_575A":       ,
    "credacc_maxhisbal_375A":     ,
    "credacc_minhisbal_90A":      ,
    "credamount_590A":            ,
    "currdebt_94A":               ,
    "downpmt_134A":               ,
    "mainoccupationinc_437A":     ,
    "maxdpdtolerance_577P":       ,
    "outstandingdebt_522A":       ,
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col)))
```

### Ações Silver — Colunas `*M` com valor sentinela

```python
code_cols_with_sentinel = [
    "cancelreason_3545846M", "district_544M", "education_1138M",
    "postype_4733339M", "profession_152M", "rejectreason_755M",
    "rejectreasonclient_4145042M"
]

for c in code_cols_with_sentinel:
    df = df.withColumn(c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c)))
```

### Ações Silver — checks_silver.py

```python
# Unicidade da chave composta
dup_count = df.groupBy("case_id", "num_group1").count().filter(F.col("count") > 1).count()
assert dup_count == 0, "train_applprev_1: duplicatas na chave (case_id, num_group1)"

# Alertar valores fora do padrão remanescentes em *M (exceto a55475b1 já tratado)
for c in ["district_544M", "profession_152M"]:
    outliers = df.filter(
        ~F.col(c).rlike(r"^[PQ]\d+_\d+_\w+$") & F.col(c).isNotNull()
    ).count()
    if outliers > 0:
        print(f"⚠️  {c}: {outliers} valor(es) fora do padrão após tratamento")
```

**Granularidade:** N linhas por `case_id` (máx. 20) — chave composta `(case_id, num_group1)`.  
**Join com train_base:** `left join` por `case_id` — cobertura parcial esperada (~1.28M de 1.526M).  
**Nulos:** todos preservados — imputação adiada para Gold.


## Dataset train_applprev_2

In [12]:
df_train_applprev_2 = spark.read.parquet(str(bronze_dir / "train_applprev_2"))

In [13]:
df_train_applprev_2.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- cacccardblochreas_147M: string (nullable = true)
 |-- conts_type_509L: string (nullable = true)
 |-- credacc_cards_status_52L: string (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- num_group2: long (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [14]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_train_applprev_2.count()

null_counts = df_train_applprev_2.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_train_applprev_2.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_train_applprev_2.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 14075487

coluna                                          null_count   null_pct
----------------------------------------------------------------------
credacc_cards_status_52L                          13733404     97.57%
conts_type_509L                                    2394056     17.01%
cacccardblochreas_147M                              109249      0.78%
case_id                                                  0       0.0%
num_group1                                               0       0.0%
num_group2                                               0       0.0%
_run_id                                                  0       0.0%
_ingest_date                                             0       0.0%
_source_file                                             0       0.0%


In [15]:
# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_train_applprev_2.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_train_applprev_2.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Strings ===
  cacccardblochreas_147M: {'a55475b1': 13958443}
  conts_type_509L: nenhum sentinela detectado
  credacc_cards_status_52L: nenhum sentinela detectado


In [16]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_train_applprev_2.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_train_applprev_2.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
case_id                                               1285631
num_group1                                                 20
num_group2                                                 12
cacccardblochreas_147M                                      9
conts_type_509L                                             9
credacc_cards_status_52L                                    6


In [17]:
# ── CHAVE COMPOSTA (case_id + num_group1 + num_group2) ───────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===")

unique_keys = df_train_applprev_2.select("case_id", "num_group1", "num_group2").distinct().count()
dups        = total - unique_keys

print(f"  Total de linhas          : {total}")
print(f"  Chaves únicas            : {unique_keys}")
print(f"  Duplicatas               : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1, num_group2) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_train_applprev_2.groupBy("case_id", "num_group1", "num_group2").count() \
                       .filter(F.col("count") > 1) \
                       .orderBy(F.col("count").desc()) \
                       .show(10, truncate=False)

# Distribuição de num_group1 e num_group2
print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_train_applprev_2.groupBy("num_group1").count() \
                   .orderBy("num_group1") \
                   .show(25, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group2 ===")
df_train_applprev_2.groupBy("num_group2").count() \
                   .orderBy("num_group2") \
                   .show(25, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===
  Total de linhas          : 14075487
  Chaves únicas            : 14075487
  Duplicatas               : 0
  ✅ (case_id, num_group1, num_group2) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+-------+
|num_group1|count  |
+----------+-------+
|0         |2276307|
|1         |1976391|
|2         |1685107|
|3         |1421703|
|4         |1190551|
|5         |990136 |
|6         |820840 |
|7         |680934 |
|8         |563599 |
|9         |467994 |
|10        |389190 |
|11        |324193 |
|12        |271808 |
|13        |227714 |
|14        |191417 |
|15        |161700 |
|16        |136458 |
|17        |115979 |
|18        |98875  |
|19        |84591  |
+----------+-------+


=== DISTRIBUIÇÃO DE num_group2 ===
+----------+-------+
|num_group2|count  |
+----------+-------+
|0         |6525978|
|1         |4976173|
|2         |2286939|
|3         |276223 |
|4         |9394   |
|5         |717

In [18]:
# ── PADRÃO DAS COLUNAS CODIFICADAS (*M) ──────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = ["cacccardblochreas_147M"]

for c in code_cols:
    outliers = df_train_applprev_2.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") & F.col(c).isNotNull()
    ).select(c).distinct()
    count = outliers.count()
    if count > 0:
        print(f"  ⚠️  {c}: {count} valor(es) fora do padrão")
        outliers.show(10, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  cacccardblochreas_147M: 1 valor(es) fora do padrão
+----------------------+
|cacccardblochreas_147M|
+----------------------+
|a55475b1              |
+----------------------+



## 📋 Análise — `train_applprev_2`

### Schema e Tipos

| Coluna                   | dtype Bronze | Ação Silver        | Justificativa                              |
|--------------------------|--------------|--------------------|--------------------------------------------|
| case_id                  | long         | manter             | Parte da chave composta, tipo correto      |
| num_group1               | long         | cast → IntegerType | Range 0–19, long desnecessário             |
| num_group2               | long         | cast → IntegerType | Range 0–11, long desnecessário             |
| cacccardblochreas_147M   | string       | manter             | Código ofuscado, string correto            |
| conts_type_509L          | string       | manter             | Categórico, string correto                 |
| credacc_cards_status_52L | string       | manter             | Categórico, string correto                 |
| _run_id                  | string       | propagar           | Metadado Bronze, não alterar               |
| _ingest_date             | date         | propagar           | Metadado Bronze, não alterar               |
| _source_file             | string       | propagar           | Metadado Bronze, não alterar               |

### Nulos

| Classificação | Coluna | null_pct | Ação Silver |
|---|---|---|---|
| Estrutural | `credacc_cards_status_52L` | 97.57% | Manter null — só existe quando há cartão associado |
| Informativo | `conts_type_509L` | 17.01% | Manter null — ausência tem significado de negócio |
| Residual | `cacccardblochreas_147M` | 0.78% | Manter null — após tratamento do sentinela `a55475b1` o null_pct real será maior (ver sentinelas) |

⚠️ Nenhuma imputação na Silver — todos os nulos preservados.

### Sentinelas

`cacccardblochreas_147M` apresenta 13.958.443 ocorrências do valor `a55475b1` — padrão recorrente no dataset como sentinela de dado ausente/mascarado (mesmo valor detectado em `train_applprev_1`). Substituir por `null` na Silver.

⚠️ Após a substituição, o null_pct real de `cacccardblochreas_147M` será de aproximadamente **99.77%** (109.249 nulos originais + 13.958.443 sentinelas = ~14.067.692 de 14.075.487 linhas).

### Cardinalidade
- `case_id`: ~1.28M distintos — mesma cobertura de `train_applprev_1`, esperado
- `cacccardblochreas_147M`: 9 valores distintos (incluindo `a55475b1`) — após tratamento, 8 categorias válidas
- `conts_type_509L`: 9 valores distintos — baixa cardinalidade, categórico simples
- `credacc_cards_status_52L`: 6 valores distintos — baixa cardinalidade, categórico simples
- `num_group1`: 20 distintos (0–19) — alinhado com `train_applprev_1`
- `num_group2`: 12 distintos (0–11) — sub-nível dentro de cada aplicação anterior

### Chave Composta
`(case_id, num_group1, num_group2)` é chave composta única (0 duplicatas em 14.075.487 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição
Tabela com **três níveis de granularidade**: `case_id` → `num_group1` (aplicação anterior) → `num_group2` (cartão dentro da aplicação).

- `num_group1`: distribuição decrescente de 0 a 19 — mesmo padrão de `train_applprev_1`, esperado
- `num_group2`: fortemente concentrado em 0 e 1 (96% das linhas); valores acima de 4 são raros (< 800 ocorrências combinadas) — a maioria das aplicações anteriores tem no máximo 2 cartões associados

### Colunas Codificadas (`*M`) — Valores Fora do Padrão
`cacccardblochreas_147M` contém o valor `a55475b1` — mesmo sentinela recorrente identificado em `train_applprev_1`. Tratado via substituição por `null` (ver ação abaixo).

---

### Ações Silver — Casts

```python
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))
df = df.withColumn("num_group2", F.col("num_group2").cast("int"))
```

### Ações Silver — Sentinelas → null

```python
df = df.withColumn(
    "cacccardblochreas_147M",
    F.when(F.col("cacccardblochreas_147M") == "a55475b1", None)
     .otherwise(F.col("cacccardblochreas_147M"))
)
```

### Ações Silver — checks_silver.py

```python
# Unicidade da chave composta
dup_count = df.groupBy("case_id", "num_group1", "num_group2").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_applprev_2: duplicatas na chave (case_id, num_group1, num_group2)"
```

**Granularidade:** três níveis — `(case_id, num_group1, num_group2)`.  
**Join com train_base:** `left join` por `case_id` — cobertura parcial esperada (~1.28M de 1.526M).  
**Nulos:** todos preservados — imputação adiada para Gold.  
**Observação:** `a55475b1` é sentinela recorrente no dataset Home Credit — aplicar o mesmo tratamento em todas as tabelas onde for detectado.


## Dataset train_credit_bureau_a_1

In [5]:
from pathlib import Path

# Sobe dois níveis a partir do notebook (notebooks/ → raiz do projeto)
repo_root  = Path().resolve().parent
bronze_dir = repo_root / "data" / "bronze"

print(f"bronze_dir: {bronze_dir}")  # confirme o path antes de rodar

df_credit_bureau_a_1 = load_logical_dataset(spark, bronze_dir, "train_credit_bureau_a_1")

bronze_dir: C:\Users\fred\Documents\Estudo de dados\Projeto\credit-risk\data\bronze
Partições encontradas para 'train_credit_bureau_a_1':
  train_credit_bureau_a_1_0
  train_credit_bureau_a_1_1
  train_credit_bureau_a_1_2
  train_credit_bureau_a_1_3


In [27]:
df_credit_bureau_a_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- annualeffectiverate_199L: double (nullable = true)
 |-- annualeffectiverate_63L: double (nullable = true)
 |-- classificationofcontr_13M: string (nullable = true)
 |-- classificationofcontr_400M: string (nullable = true)
 |-- contractst_545M: string (nullable = true)
 |-- contractst_964M: string (nullable = true)
 |-- contractsum_5085717L: double (nullable = true)
 |-- credlmt_230A: double (nullable = true)
 |-- credlmt_935A: double (nullable = true)
 |-- dateofcredend_289D: string (nullable = true)
 |-- dateofcredend_353D: string (nullable = true)
 |-- dateofcredstart_181D: string (nullable = true)
 |-- dateofcredstart_739D: string (nullable = true)
 |-- dateofrealrepmt_138D: string (nullable = true)
 |-- debtoutstand_525A: double (nullable = true)
 |-- debtoverdue_47A: double (nullable = true)
 |-- description_351M: string (nullable = true)
 |-- dpdmax_139P: double (nullable = true)
 |-- dpdmax_757P: double (nullable = true)
 |-- dpdmaxd

In [28]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_credit_bureau_a_1.count()

null_counts = df_credit_bureau_a_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_credit_bureau_a_1.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_credit_bureau_a_1.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 15940537

coluna                                          null_count   null_pct
----------------------------------------------------------------------
prolongationcount_599L                            15901589     99.76%
interestrate_508L                                 15874717     99.59%
annualeffectiverate_63L                           15629294     98.05%
contractsum_5085717L                              15621206      98.0%
prolongationcount_1120L                           15459068     96.98%
instlamount_852A                                  15346371     96.27%
numberofoverdueinstlmaxdat_641D                   15260655     95.73%
overdueamountmax2date_1142D                       15254776      95.7%
annualeffectiverate_199L                          15213904     95.44%
residualamount_488A                               15052745     94.43%
credlmt_230A                                      15044115     94.38%
nominalrate_281L                                  14960980    

In [29]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, 9999999, -1]

numeric_cols = [c for c, t in df_credit_bureau_a_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1")]

for c in numeric_cols:
    counts = df_credit_bureau_a_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_credit_bureau_a_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_credit_bureau_a_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
  annualeffectiverate_199L: {99: 36}
  contractsum_5085717L: {9999: 2}
  credlmt_935A: {99: 1, 999: 1, 9999: 1}
  debtoutstand_525A: {99: 1, 999: 6, 9999: 6}
  debtoverdue_47A: {99: 1, 9999: 1}
  dpdmax_139P: {99: 157, 999: 16}
  dpdmax_757P: {99: 1215, 999: 211, -1: 655}
  instlamount_768A: {99: 2, 999: 33, 9999: 6}
  instlamount_852A: {99: 2, 999: 9, 9999: 1}
  monthlyinstlamount_332A: {99: 2, 999: 93, 9999: 23, 99999: 1}
  monthlyinstlamount_674A: {99: 9, 999: 93, 9999: 18, 99999: 1}
  nominalrate_498L: {99: 36}
  numberofcontrsvalue_258L: {99: 1}
  numberofcontrsvalue_358L: {99: 2}
  numberofinstls_229L: {99: 124}
  numberofinstls_320L: {99: 186}
  numberofoutstandinstls_59L: {99: 489}
  numberofoverdueinstlmax_1039L: {99: 196, 999: 10}
  numberofoverdueinstlmax_1151L: {99: 1366, 999: 332}
  numberofoverdueinstls_725L: {99: 42, 999: 7}
  outstandingamount_354A: {99: 1}
  outstandingamount_362A: {999: 1, 9999: 6}
  overdueamount_31A: {99: 1}
  overduea

In [30]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_credit_bureau_a_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_credit_bureau_a_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
case_id                                               1340900
outstandingamount_362A                                1315023
totaloutstanddebtvalue_39A                            1203061
debtoutstand_525A                                     1180860
overdueamountmax2_398A                                 967318
overdueamountmax_35A                                   887585
monthlyinstlamount_674A                                880310
residualamount_856A                                    530214
totalamount_6A                                         483170
monthlyinstlamount_332A                                470524
overdueamountmax2_14A                                  457750
overdueamountmax_155A                                  400753
totalamount_996A                                       280511
contractsum_5085717L                

In [31]:
# ── CHAVE COMPOSTA ───────────────────────────────────────────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===")

unique_keys = df_credit_bureau_a_1.select("case_id", "num_group1").distinct().count()
dups        = total - unique_keys

print(f"  Total de linhas          : {total}")
print(f"  Chaves únicas            : {unique_keys}")
print(f"  Duplicatas               : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_credit_bureau_a_1.groupBy("case_id", "num_group1").count() \
                        .filter(F.col("count") > 1) \
                        .orderBy(F.col("count").desc()) \
                        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_credit_bureau_a_1.groupBy("num_group1").count() \
                    .orderBy("num_group1") \
                    .show(50, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===
  Total de linhas          : 15940537
  Chaves únicas            : 15940537
  Duplicatas               : 0
  ✅ (case_id, num_group1) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+-------+
|num_group1|count  |
+----------+-------+
|0         |1386273|
|1         |1386081|
|2         |1385817|
|3         |1385691|
|4         |1385691|
|5         |1385691|
|6         |1385691|
|7         |1385691|
|8         |1385691|
|9         |712528 |
|10        |660550 |
|11        |319007 |
|12        |266917 |
|13        |223530 |
|14        |186904 |
|15        |156624 |
|16        |131211 |
|17        |110186 |
|18        |92891  |
|19        |78139  |
|20        |66081  |
|21        |56170  |
|22        |47816  |
|23        |40939  |
|24        |35092  |
|25        |30302  |
|26        |26103  |
|27        |22630  |
|28        |19693  |
|29        |17255  |
|30        |15199  |
|31        |13325  |
|32        |1

In [32]:
# ── RANGE DE DATAS ───────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

date_cols = [c for c, t in df_credit_bureau_a_1.dtypes
             if t == "string" and c.endswith("D")
             and c not in ("_run_id", "_source_file")]

df_credit_bureau_a_1.select([
    val
    for c in date_cols
    for val in (F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max"))
]).show(truncate=False)


=== RANGE DE DATAS ===
+----------------------+----------------------+----------------------+----------------------+------------------------+------------------------+------------------------+------------------------+------------------------+------------------------+--------------------+--------------------+-------------------+-------------------+-----------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------------------+------------------------+------------------------+
|dateofcredend_289D_min|dateofcredend_289D_max|dateofcredend_353D_min|dateofcredend_353D_max|dateofcredstart_181D_min|dateofcredstart_181D_max|dateofcredstart_739D_min|dateofcredstart_739D_max|dateofrealrepmt_138D_min|dateofrealrepmt_138D_max|lastupdate_1112D_min|lastupdate_1112D_max|lastupdate_388D_min|lastupdate_388D_max|numb

In [11]:
from pyspark.sql import functions as F

date_range_cols = [
    "dateofcredstart_181D", "dateofcredstart_739D",
    "dateofcredend_289D",   "dateofcredend_353D",
    "dateofrealrepmt_138D"
]

# Filtra apenas datas que NÃO são os valores extremos conhecidos (1900, 2098)
for c in date_range_cols:
    df_credit_bureau_a_1.filter(
        F.col(c).isNotNull() &
        (F.col(c) != "1900-01-15") &
        (F.col(c) != "2098-01-15")
    ).select(
        F.min(c).alias("min_real"),
        F.max(c).alias("max_real")
    ).withColumn("coluna", F.lit(c)) \
     .select("coluna", "min_real", "max_real") \
     .show(truncate=False)




+--------------------+----------+----------+
|coluna              |min_real  |max_real  |
+--------------------+----------+----------+
|dateofcredstart_181D|1994-12-04|2020-10-12|
+--------------------+----------+----------+

+--------------------+----------+----------+
|coluna              |min_real  |max_real  |
+--------------------+----------+----------+
|dateofcredstart_739D|1999-09-15|2021-08-13|
+--------------------+----------+----------+

+------------------+----------+----------+
|coluna            |min_real  |max_real  |
+------------------+----------+----------+
|dateofcredend_289D|2004-05-29|2058-01-14|
+------------------+----------+----------+

+------------------+----------+----------+
|coluna            |min_real  |max_real  |
+------------------+----------+----------+
|dateofcredend_353D|1999-02-24|2068-01-15|
+------------------+----------+----------+

+--------------------+----------+----------+
|coluna              |min_real  |max_real  |
+--------------------+----

In [33]:
# ── PADRÃO DAS COLUNAS CODIFICADAS (*M) ──────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [c for c in df_credit_bureau_a_1.columns if c.endswith("M")]

for c in code_cols:
    outliers = df_credit_bureau_a_1.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") & F.col(c).isNotNull()
    ).select(c).distinct()
    count = outliers.count()
    if count > 0:
        print(f"  ⚠️  {c}: {count} valor(es) fora do padrão")
        outliers.show(5, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  classificationofcontr_13M: 11 valor(es) fora do padrão
+-------------------------+
|classificationofcontr_13M|
+-------------------------+
|be7b251d                 |
|ea6782cc                 |
|1cf4e481                 |
|2c070815                 |
|00135d9c                 |
+-------------------------+
only showing top 5 rows

  ⚠️  classificationofcontr_400M: 389 valor(es) fora do padrão
+--------------------------+
|classificationofcontr_400M|
+--------------------------+
|51590aa9                  |
|acba4f13                  |
|ffee884a                  |
|fa2a66b3                  |
|01938327                  |
+--------------------------+
only showing top 5 rows

  ⚠️  contractst_545M: 51 valor(es) fora do padrão
+---------------+
|contractst_545M|
+---------------+
|83931972       |
|dd67cff0       |
|54132f86       |
|6d2380c9       |
|7640edc3       |
+---------------+
only showing top 5 rows

  ⚠️  contractst_964M: 295 valo

In [12]:
code_cols = [c for c in df_credit_bureau_a_1.columns if c.endswith("M")]

for c in code_cols:
    total_notnull = df_credit_bureau_a_1.filter(F.col(c).isNotNull()).count()
    outliers = df_credit_bureau_a_1.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull() &
        (F.col(c) != "a55475b1")
    ).count()
    pct = round(outliers / total_notnull * 100, 2) if total_notnull > 0 else 0
    print(f"  {c}: {outliers} hashes fora do padrão ({pct}% dos não-nulos)")


  classificationofcontr_13M: 2656721 hashes fora do padrão (16.67% dos não-nulos)
  classificationofcontr_400M: 6741045 hashes fora do padrão (42.29% dos não-nulos)
  contractst_545M: 2651823 hashes fora do padrão (16.64% dos não-nulos)
  contractst_964M: 6749556 hashes fora do padrão (42.34% dos não-nulos)
  description_351M: 61993 hashes fora do padrão (0.39% dos não-nulos)
  financialinstitution_382M: 4503506 hashes fora do padrão (28.25% dos não-nulos)
  financialinstitution_591M: 1322724 hashes fora do padrão (8.3% dos não-nulos)
  purposeofcred_426M: 2658858 hashes fora do padrão (16.68% dos não-nulos)
  purposeofcred_874M: 6746904 hashes fora do padrão (42.33% dos não-nulos)
  subjectrole_182M: 1301112 hashes fora do padrão (8.16% dos não-nulos)
  subjectrole_93M: 1275537 hashes fora do padrão (8.0% dos não-nulos)


In [34]:
# ── COLUNAS DUPLICADAS (*_xxxL / *_yyyL com mesma semântica) ─────────────────
# Esta tabela tem pares de colunas com nomes similares (ex: credlmt_230A / credlmt_935A)
# Verificar correlação entre pares para identificar redundância
print("=== AMOSTRA DE PARES DE COLUNAS SIMILARES ===")

df_credit_bureau_a_1.select(
    "credlmt_230A", "credlmt_935A",
    "dpdmax_139P", "dpdmax_757P",
    "annualeffectiverate_199L", "annualeffectiverate_63L"
).show(10, truncate=False)


=== AMOSTRA DE PARES DE COLUNAS SIMILARES ===
+------------+------------+-----------+-----------+------------------------+-----------------------+
|credlmt_230A|credlmt_935A|dpdmax_139P|dpdmax_757P|annualeffectiverate_199L|annualeffectiverate_63L|
+------------+------------+-----------+-----------+------------------------+-----------------------+
|NULL        |135806.0    |0.0        |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |0.0        |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL       |NULL                    |NULL                   |
|NULL        |NULL        |NULL       |NULL  

In [13]:
from pyspark.sql import functions as F

pairs = [
    ("credlmt_230A",              "credlmt_935A"),
    ("dpdmax_139P",               "dpdmax_757P"),
    ("annualeffectiverate_199L",  "annualeffectiverate_63L"),
    ("instlamount_768A",          "instlamount_852A"),
    ("monthlyinstlamount_332A",   "monthlyinstlamount_674A"),
    ("numberofinstls_229L",       "numberofinstls_320L"),
    ("totalamount_6A",            "totalamount_996A"),
]

print(f"{'Par':<55} {'ambos_preenchidos':>18} {'só_A':>10} {'só_B':>10}")
print("-" * 95)

for a, b in pairs:
    both = df_credit_bureau_a_1.filter(
        F.col(a).isNotNull() & F.col(b).isNotNull()
    ).count()
    only_a = df_credit_bureau_a_1.filter(
        F.col(a).isNotNull() & F.col(b).isNull()
    ).count()
    only_b = df_credit_bureau_a_1.filter(
        F.col(a).isNull() & F.col(b).isNotNull()
    ).count()
    print(f"{a+' / '+b:<55} {both:>18} {only_a:>10} {only_b:>10}")


Par                                                      ambos_preenchidos       só_A       só_B
-----------------------------------------------------------------------------------------------
credlmt_230A / credlmt_935A                                         162430     733992    1092555
dpdmax_139P / dpdmax_757P                                          1762039     881997    4764856
annualeffectiverate_199L / annualeffectiverate_63L                   17750     708883     293493
instlamount_768A / instlamount_852A                                  81266    1157928     512900
monthlyinstlamount_332A / monthlyinstlamount_674A                  1570564    1072078    4721603
numberofinstls_229L / numberofinstls_320L                           800438    5052700     603427
totalamount_6A / totalamount_996A                                   803012    5057869     601279


## 📋 Análise — `train_credit_bureau_a_1` (union de `_0` a `_3`)



### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| num_group1 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| annualeffectiverate_199L | double | manter | Taxa percentual, double correto |
| annualeffectiverate_63L | double | manter | Taxa percentual, double correto |
| classificationofcontr_13M | string | manter | Código ofuscado, string correto |
| classificationofcontr_400M | string | manter | Código ofuscado, string correto |
| contractst_545M | string | manter | Código ofuscado, string correto |
| contractst_964M | string | manter | Código ofuscado, string correto |
| contractsum_5085717L | double | manter | Valor monetário, double correto |
| credlmt_230A | double | manter | Limite de crédito, double correto |
| credlmt_935A | double | manter | Limite de crédito, double correto |
| dateofcredend_289D | string | cast → DateType | Data armazenada como string |
| dateofcredend_353D | string | cast → DateType | Data armazenada como string |
| dateofcredstart_181D | string | cast → DateType | Data armazenada como string |
| dateofcredstart_739D | string | cast → DateType | Data armazenada como string |
| dateofrealrepmt_138D | string | cast → DateType | Data armazenada como string |
| debtoutstand_525A | double | manter | Valor monetário, double correto |
| debtoverdue_47A | double | manter | Valor monetário, double correto |
| description_351M | string | manter | Código ofuscado, string correto |
| dpdmax_139P | double | manter | Métrica de atraso, double correto |
| dpdmax_757P | double | manter | Métrica de atraso, double correto |
| dpdmaxdatemonth_442T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| dpdmaxdatemonth_89T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| dpdmaxdateyear_596T | double | cast → IntegerType | Ano, double desnecessário |
| dpdmaxdateyear_896T | double | cast → IntegerType | Ano, double desnecessário |
| financialinstitution_382M | string | manter | Código ofuscado, string correto |
| financialinstitution_591M | string | manter | Código ofuscado, string correto |
| instlamount_768A | double | manter | Valor monetário, double correto |
| instlamount_852A | double | manter | Valor monetário, double correto |
| interestrate_508L | double | manter | Taxa percentual, double correto |
| lastupdate_1112D | string | cast → DateType | Data armazenada como string |
| lastupdate_388D | string | cast → DateType | Data armazenada como string |
| monthlyinstlamount_332A | double | manter | Valor monetário, double correto |
| monthlyinstlamount_674A | double | manter | Valor monetário, double correto |
| nominalrate_281L | double | manter | Taxa percentual, double correto |
| nominalrate_498L | double | manter | Taxa percentual, double correto |
| numberofcontrsvalue_258L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofcontrsvalue_358L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofinstls_229L | double | cast → IntegerType | Contagem de parcelas, double desnecessário |
| numberofinstls_320L | double | cast → IntegerType | Contagem de parcelas, double desnecessário |
| numberofoutstandinstls_520L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoutstandinstls_59L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoverdueinstlmax_1039L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoverdueinstlmax_1151L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoverdueinstlmaxdat_148D | string | cast → DateType | Data armazenada como string |
| numberofoverdueinstlmaxdat_641D | string | cast → DateType | Data armazenada como string |
| numberofoverdueinstls_725L | double | cast → IntegerType | Contagem, double desnecessário |
| numberofoverdueinstls_834L | double | cast → IntegerType | Contagem, double desnecessário |
| outstandingamount_354A | double | manter | Valor monetário, double correto |
| outstandingamount_362A | double | manter | Valor monetário, double correto |
| overdueamount_31A | double | manter | Valor monetário, double correto |
| overdueamount_659A | double | manter | Valor monetário, double correto |
| overdueamountmax2_14A | double | manter | Valor monetário, double correto |
| overdueamountmax2_398A | double | manter | Valor monetário, double correto |
| overdueamountmax2date_1002D | string | cast → DateType | Data armazenada como string |
| overdueamountmax2date_1142D | string | cast → DateType | Data armazenada como string |
| overdueamountmax_155A | double | manter | Valor monetário, double correto |
| overdueamountmax_35A | double | manter | Valor monetário, double correto |
| overdueamountmaxdatemonth_284T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| overdueamountmaxdatemonth_365T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| overdueamountmaxdateyear_2T | double | cast → IntegerType | Ano, double desnecessário |
| overdueamountmaxdateyear_994T | double | cast → IntegerType | Ano, double desnecessário |
| periodicityofpmts_1102L | double | cast → IntegerType | 5 valores distintos — periodicidade categórica |
| periodicityofpmts_837L | double | cast → IntegerType | 5 valores distintos — periodicidade categórica |
| prolongationcount_1120L | double | cast → IntegerType | Contagem de prorrogações, double desnecessário |
| prolongationcount_599L | double | cast → IntegerType | Contagem de prorrogações, double desnecessário |
| purposeofcred_426M | string | manter | Código ofuscado, string correto |
| purposeofcred_874M | string | manter | Código ofuscado, string correto |
| refreshdate_3813885D | string | cast → DateType | Data armazenada como string |
| residualamount_488A | double | manter | Valor monetário, double correto |
| residualamount_856A | double | manter | Valor monetário, double correto |
| subjectrole_182M | string | manter | Código ofuscado, string correto |
| subjectrole_93M | string | manter | Código ofuscado, string correto |
| totalamount_6A | double | manter | Valor monetário, double correto |
| totalamount_996A | double | manter | Valor monetário, double correto |
| totaldebtoverduevalue_178A | double | manter | Valor monetário, double correto |
| totaldebtoverduevalue_718A | double | manter | Valor monetário, double correto |
| totaloutstanddebtvalue_39A | double | manter | Valor monetário, double correto |
| totaloutstanddebtvalue_668A | double | manter | Valor monetário, double correto |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

### Nulos

Tabela com alto volume de nulos estruturais organizados em **grupos funcionais** — colunas do mesmo grupo têm `null_pct` idêntico, indicando que só existem juntas:

| Grupo | null_pct | Colunas representativas |
|---|---|---|
| Grupo A | ~99% | `prolongationcount_599L`, `interestrate_508L` |
| Grupo B | ~96–98% | `annualeffectiverate_*`, `contractsum_*`, `instlamount_852A` |
| Grupo C | ~91–94% | `credlmt_*`, `debtoutstand_*`, `numberofcontrsvalue_358L` e adjacentes |
| Grupo D | ~83% | `dpdmax_139P`, `dateofcredend_289D`, `numberofoverdueinstls_725L` e adjacentes |
| Grupo E | ~57–63% | `dateofcredend_353D`, `dateofcredstart_181D`, `dpdmax_757P` e adjacentes |
| Baixo nulo | ~30% | `refreshdate_3813885D` |
| Zero nulo | 0% | `case_id`, `num_group1`, todas as colunas `*M`, metadados |

⚠️ Nenhuma imputação na Silver — todos os nulos preservados. Imputação adiada para Gold.

### Sentinelas

**Numéricos** — 35 colunas afetadas. Substituir por `null` na Silver:

```python
sentinel_map = {
    "annualeffectiverate_199L":      ,
    "contractsum_5085717L":          ,
    "credlmt_935A":                  ,
    "debtoutstand_525A":             ,
    "debtoverdue_47A":               ,
    "dpdmax_139P":                   ,
    "dpdmax_757P":                   [99, 999, -1],
    "instlamount_768A":              ,
    "instlamount_852A":              ,
    "monthlyinstlamount_332A":       ,
    "monthlyinstlamount_674A":       ,
    "nominalrate_498L":              ,
    "numberofcontrsvalue_258L":      ,
    "numberofcontrsvalue_358L":      ,
    "numberofinstls_229L":           ,
    "numberofinstls_320L":           ,
    "numberofoutstandinstls_59L":    ,
    "numberofoverdueinstlmax_1039L": ,
    "numberofoverdueinstlmax_1151L": ,
    "numberofoverdueinstls_725L":    ,
    "outstandingamount_354A":        ,
    "outstandingamount_362A":        ,
    "overdueamount_31A":             ,
    "overdueamount_659A":            ,
    "overdueamountmax2_14A":         ,
    "overdueamountmax2_398A":        ,
    "overdueamountmax_155A":         ,
    "overdueamountmax_35A":          ,
    "residualamount_856A":           ,
    "totalamount_6A":                ,
    "totalamount_996A":              ,
    "totaldebtoverduevalue_178A":    ,
    "totaldebtoverduevalue_718A":    ,
    "totaloutstanddebtvalue_39A":    ,
    "totaloutstanddebtvalue_668A":   ,
}
```

⚠️ `dpdmax_757P` é o único campo com sentinela `-1` identificado no projeto até agora.

**Strings** — `a55475b1` detectado em todas as colunas `*M`. Substituir por `null`.

### Cardinalidade
- `case_id`: ~1.34M distintos — cobertura parcial do `train_base` (1.526M), esperado para bureau
- `num_group1`: 518 valores distintos — granularidade muito maior que `train_applprev` (20); representa múltiplos contratos de bureau por cliente
- Colunas monetárias (`outstandingamount_362A`, `totaloutstanddebtvalue_39A`): alta cardinalidade — valores contínuos, esperado
- `residualamount_488A`: 9 valores distintos com 94.43% de nulos — valores monetários legítimos, nenhum sentinela

### Chave Composta
`(case_id, num_group1)` é chave composta única (0 duplicatas em 15.940.537 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição de `num_group1`
Dois regimes distintos identificados:
- **Grupos 0–8**: volume estável ~1.385.691 por grupo — indica 9 tipos fixos de produto/contrato presentes para todos os clientes com bureau
- **Grupo 9 em diante**: queda brusca e decrescente até grupo 517 — registros adicionais para clientes com histórico extenso

⚠️ `num_group1` não é apenas índice sequencial nessa tabela — os primeiros 9 grupos têm semântica fixa. Investigar na Gold se grupos 0–8 representam tipos de produto distintos.

### Range de Datas
Datas com sentinelas conhecidos (`1900-01-15` e `2098-01-15`) detectados em 3 colunas:

| Coluna | Ocorrências anômalas | Range real válido |
|---|---|---|
| `dateofcredend_289D` | 53.722 | 2004-05-29 a 2058-01-14 |
| `dateofcredend_353D` | 20.415 | 1999-02-24 a 2068-01-15 |
| `dateofrealrepmt_138D` | 230 | 2000-01-15 a 2037-08-29 |

⚠️ Datas futuras (`2058`, `2068`) são **legítimas** — contratos de longo prazo (financiamentos, hipotecas). Substituir apenas os sentinelas `1900-01-15` e `2098-01-15` por `null`.

Demais colunas de data dentro de ranges plausíveis. Todos os casts seguros após tratamento de sentinelas.

### Colunas Codificadas (`*M`) — Dois Padrões de Codificação
Diferente das tabelas anteriores, as colunas `*M` desta tabela apresentam **dois esquemas de codificação legítimos**:
- **Padrão `P\d+_\d+_\d+`** — codificação principal
- **Hashes hexadecimais** (`be7b251d`, `ea6782cc`) — codificação alternativa, provavelmente de fonte de bureau distinta

| Coluna | % hashes dos não-nulos |
|---|---|
| `classificationofcontr_400M` | 42.29% |
| `contractst_964M` | 42.34% |
| `purposeofcred_874M` | 42.33% |
| `financialinstitution_382M` | 28.25% |
| `classificationofcontr_13M` | 16.67% |
| `contractst_545M` | 16.64% |
| `purposeofcred_426M` | 16.68% |
| `financialinstitution_591M` | 8.30% |
| `subjectrole_182M` / `subjectrole_93M` | ~8% |
| `description_351M` | 0.39% |

**Ação Silver:** manter ambos os padrões como string. Substituir apenas `a55475b1` por `null`. Registrar presença dos dois padrões no `checks_silver.py`.

### Pares de Colunas Similares
Todas as colunas similares têm sobreposição relevante — **não são mutuamente exclusivas**:

| Par | Ambos preenchidos | Estratégia Gold |
|---|---|---|
| `dpdmax_139P` / `dpdmax_757P` | 1.762.039 | Features independentes (períodos distintos) |
| `monthlyinstlamount_332A` / `monthlyinstlamount_674A` | 1.570.564 | Features independentes |
| `numberofinstls_229L` / `numberofinstls_320L` | 800.438 | Features independentes |
| `totalamount_6A` / `totalamount_996A` | 803.012 | Features independentes |
| `credlmt_230A` / `credlmt_935A` | 162.430 | Avaliar `coalesce(A, B)` |
| `instlamount_768A` / `instlamount_852A` | 81.266 | Avaliar `coalesce(A, B)` |
| `annualeffectiverate_199L` / `annualeffectiverate_63L` | 17.750 | Avaliar `coalesce(A, B)` |

**Ação Silver:** manter todos os pares intactos. Consolidação adiada para Gold.

---

### Ações Silver — Casts

```python
date_cols = [
    "dateofcredend_289D", "dateofcredend_353D", "dateofcredstart_181D",
    "dateofcredstart_739D", "dateofrealrepmt_138D", "lastupdate_1112D",
    "lastupdate_388D", "numberofoverdueinstlmaxdat_148D",
    "numberofoverdueinstlmaxdat_641D", "overdueamountmax2date_1002D",
    "overdueamountmax2date_1142D", "refreshdate_3813885D"
]
for c in date_cols:
    df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))

int_cols = [
    "num_group1", "dpdmaxdatemonth_442T", "dpdmaxdatemonth_89T",
    "dpdmaxdateyear_596T", "dpdmaxdateyear_896T",
    "numberofcontrsvalue_258L", "numberofcontrsvalue_358L",
    "numberofinstls_229L", "numberofinstls_320L",
    "numberofoutstandinstls_520L", "numberofoutstandinstls_59L",
    "numberofoverdueinstlmax_1039L", "numberofoverdueinstlmax_1151L",
    "numberofoverdueinstls_725L", "numberofoverdueinstls_834L",
    "overdueamountmaxdatemonth_284T", "overdueamountmaxdatemonth_365T",
    "overdueamountmaxdateyear_2T", "overdueamountmaxdateyear_994T",
    "periodicityofpmts_1102L", "periodicityofpmts_837L",
    "prolongationcount_1120L", "prolongationcount_599L"
]
for c in int_cols:
    df = df.withColumn(c, F.col(c).cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — Sentinelas em Datas → null

```python
sentinel_dates = ["1900-01-15", "2098-01-15"]
date_sentinel_cols = [
    "dateofcredend_289D", "dateofcredend_353D",
    "dateofcredstart_181D", "dateofcredstart_739D",
    "dateofrealrepmt_138D"
]
for c in date_sentinel_cols:
    df = df.withColumn(
        c, F.when(F.col(c).isin(sentinel_dates), None).otherwise(F.col(c))
    )
```

### Ações Silver — Sentinelas em Colunas `*M` → null

```python
m_cols = [c for c in df.columns if c.endswith("M")]
for c in m_cols:
    df = df.withColumn(
        c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c))
    )
```

### Ações Silver — checks_silver.py

```python
# Unicidade da chave composta
dup_count = df.groupBy("case_id", "num_group1").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_credit_bureau_a_1: duplicatas na chave (case_id, num_group1)"

# Auditoria de padrões em colunas *M
for c in [col for col in df.columns if col.endswith("M")]:
    total_notnull = df.filter(F.col(c).isNotNull()).count()
    pattern_p   = df.filter(F.col(c).rlike(r"^P\d+_\d+_\d+$")).count()
    pattern_hex = df.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull()
    ).count()
    print(f"{c}: padrão P={pattern_p} | hex={pattern_hex} | total={total_notnull}")
```

**Granularidade:** N linhas por `case_id` (máx. 517+) — chave composta `(case_id, num_group1)`.  
**Join com train_base:** `left join` por `case_id` — cobertura parcial esperada (~1.34M de 1.526M).  
**Nulos:** todos preservados — imputação adiada para Gold.  
**Pares similares:** mantidos intactos na Silver — decisão de consolidação (`coalesce` vs features independentes) adiada para Gold.  
**`num_group1` grupos 0–8:** semântica fixa suspeita — investigar na Gold se representam tipos de produto distintos.


## Dataset train_credit_bureau_a_2

In [14]:
df_credit_bureau_a_2 = load_logical_dataset(spark, bronze_dir, "train_credit_bureau_a_2")

Partições encontradas para 'train_credit_bureau_a_2':
  train_credit_bureau_a_2_0
  train_credit_bureau_a_2_1
  train_credit_bureau_a_2_10
  train_credit_bureau_a_2_2
  train_credit_bureau_a_2_3
  train_credit_bureau_a_2_4
  train_credit_bureau_a_2_5
  train_credit_bureau_a_2_6
  train_credit_bureau_a_2_7
  train_credit_bureau_a_2_8
  train_credit_bureau_a_2_9


In [15]:
df_credit_bureau_a_2.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- collater_typofvalofguarant_298M: string (nullable = true)
 |-- collater_typofvalofguarant_407M: string (nullable = true)
 |-- collater_valueofguarantee_1124L: double (nullable = true)
 |-- collater_valueofguarantee_876L: double (nullable = true)
 |-- collaterals_typeofguarante_359M: string (nullable = true)
 |-- collaterals_typeofguarante_669M: string (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- num_group2: long (nullable = true)
 |-- pmts_dpd_1073P: double (nullable = true)
 |-- pmts_dpd_303P: double (nullable = true)
 |-- pmts_month_158T: double (nullable = true)
 |-- pmts_month_706T: double (nullable = true)
 |-- pmts_overdue_1140A: double (nullable = true)
 |-- pmts_overdue_1152A: double (nullable = true)
 |-- pmts_year_1139T: double (nullable = true)
 |-- pmts_year_507T: double (nullable = true)
 |-- subjectroles_name_541M: string (nullable = true)
 |-- subjectroles_name_838M: string (nullable = true)
 |-- _run_id: s

In [16]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_credit_bureau_a_2.count()

null_counts = df_credit_bureau_a_2.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_credit_bureau_a_2.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_credit_bureau_a_2.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 188298452

coluna                                          null_count   null_pct
----------------------------------------------------------------------
collater_valueofguarantee_1124L                  185545117     98.54%
collater_valueofguarantee_876L                   181414202     96.34%
pmts_dpd_1073P                                   152991152     81.25%
pmts_overdue_1140A                               152850520     81.17%
pmts_month_158T                                  122168636     64.88%
pmts_year_1139T                                  122168636     64.88%
pmts_dpd_303P                                    113465446     60.26%
pmts_overdue_1152A                               113377640     60.21%
pmts_month_706T                                   29568980      15.7%
pmts_year_507T                                    29568980      15.7%
case_id                                                  0       0.0%
collater_typofvalofguarant_298M                          0   

In [17]:
# ── SENTINELAS — Numéricos ───────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, -1]

numeric_cols = [c for c, t in df_credit_bureau_a_2.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_credit_bureau_a_2.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

# ── SENTINELAS — Strings ─────────────────────────────────────────────────────
print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_credit_bureau_a_2.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_credit_bureau_a_2.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
  collater_valueofguarantee_876L: {9999: 3, 99999: 7}
  pmts_dpd_1073P: {99: 1144, 999: 207}
  pmts_dpd_303P: {99: 9490, 999: 2467, -1: 7489}
  pmts_overdue_1140A: {99: 51, 999: 7, 9999: 5}
  pmts_overdue_1152A: {99: 326, 999: 121, 9999: 27}

=== SENTINELAS — Strings ===
  collater_typofvalofguarant_298M: {'a55475b1': 185545117}
  collater_typofvalofguarant_407M: {'a55475b1': 181414202}
  collaterals_typeofguarante_359M: {'a55475b1': 181414626}
  collaterals_typeofguarante_669M: {'a55475b1': 185545250}
  subjectroles_name_541M: {'a55475b1': 181487794}
  subjectroles_name_838M: {'a55475b1': 185621035}


In [18]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_credit_bureau_a_2.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_credit_bureau_a_2.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
pmts_overdue_1152A                                    3343303
case_id                                               1340900
pmts_overdue_1140A                                    1125902
collater_valueofguarantee_876L                         158409
collater_valueofguarantee_1124L                         85988
pmts_dpd_303P                                            4372
pmts_dpd_1073P                                           4090
num_group1                                                335
num_group2                                                104
pmts_year_507T                                             27
collaterals_typeofguarante_359M                            14
collaterals_typeofguarante_669M                            14
pmts_year_1139T                                            13
pmts_month_158T                     

In [21]:
print("=== VALORES DISTINTOS — pmts_year_507T ===")
df_credit_bureau_a_2.select("pmts_year_507T") \
                    .filter(F.col("pmts_year_507T").isNotNull()) \
                    .distinct() \
                    .orderBy("pmts_year_507T") \
                    .show(30, truncate=False)

print("=== VALORES DISTINTOS — pmts_year_1139T ===")
df_credit_bureau_a_2.select("pmts_year_1139T") \
                    .filter(F.col("pmts_year_1139T").isNotNull()) \
                    .distinct() \
                    .orderBy("pmts_year_1139T") \
                    .show(20, truncate=False)


=== VALORES DISTINTOS — pmts_year_507T ===
+--------------+
|pmts_year_507T|
+--------------+
|2000.0        |
|2001.0        |
|2002.0        |
|2003.0        |
|2004.0        |
|2005.0        |
|2006.0        |
|2007.0        |
|2008.0        |
|2009.0        |
|2010.0        |
|2011.0        |
|2012.0        |
|2013.0        |
|2014.0        |
|2015.0        |
|2016.0        |
|2017.0        |
|2018.0        |
|2019.0        |
|2020.0        |
|2021.0        |
|2025.0        |
|2026.0        |
|2027.0        |
|2028.0        |
+--------------+

=== VALORES DISTINTOS — pmts_year_1139T ===
+---------------+
|pmts_year_1139T|
+---------------+
|2008.0         |
|2009.0         |
|2010.0         |
|2012.0         |
|2013.0         |
|2014.0         |
|2015.0         |
|2016.0         |
|2017.0         |
|2018.0         |
|2019.0         |
|2020.0         |
|2021.0         |
+---------------+



In [19]:
# ── CHAVE COMPOSTA (case_id + num_group1 + num_group2) ───────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===")

unique_keys = df_credit_bureau_a_2.select(
    "case_id", "num_group1", "num_group2"
).distinct().count()
dups = total - unique_keys

print(f"  Total de linhas          : {total}")
print(f"  Chaves únicas            : {unique_keys}")
print(f"  Duplicatas               : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1, num_group2) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_credit_bureau_a_2.groupBy("case_id", "num_group1", "num_group2").count() \
                        .filter(F.col("count") > 1) \
                        .orderBy(F.col("count").desc()) \
                        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_credit_bureau_a_2.groupBy("num_group1").count() \
                    .orderBy("num_group1") \
                    .show(50, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group2 ===")
df_credit_bureau_a_2.groupBy("num_group2").count() \
                    .orderBy("num_group2") \
                    .show(50, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===
  Total de linhas          : 188298452
  Chaves únicas            : 188298452
  Duplicatas               : 0
  ✅ (case_id, num_group1, num_group2) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+--------+
|num_group1|count   |
+----------+--------+
|0         |42903174|
|1         |29641063|
|2         |21502239|
|3         |16576418|
|4         |13333181|
|5         |10990028|
|6         |9107111 |
|7         |7545763 |
|8         |6239007 |
|9         |5166969 |
|10        |4246926 |
|11        |3483897 |
|12        |2864842 |
|13        |2361410 |
|14        |1933195 |
|15        |1590305 |
|16        |1309651 |
|17        |1081678 |
|18        |892716  |
|19        |745440  |
|20        |623243  |
|21        |522688  |
|22        |439851  |
|23        |370744  |
|24        |313399  |
|25        |267231  |
|26        |228871  |
|27        |197394  |
|28        |170826  |
|29        |149790

In [20]:
# ── PADRÃO DAS COLUNAS CODIFICADAS (*M) ──────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [c for c in df_credit_bureau_a_2.columns if c.endswith("M")]

for c in code_cols:
    total_notnull = df_credit_bureau_a_2.filter(F.col(c).isNotNull()).count()
    outliers = df_credit_bureau_a_2.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull() &
        (F.col(c) != "a55475b1")
    ).count()
    pct = round(outliers / total_notnull * 100, 2) if total_notnull > 0 else 0
    if outliers > 0:
        print(f"  ⚠️  {c}: {outliers} hashes fora do padrão ({pct}% dos não-nulos)")
        df_credit_bureau_a_2.filter(
            ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
            F.col(c).isNotNull() &
            (F.col(c) != "a55475b1")
        ).select(c).distinct().show(5, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  collater_typofvalofguarant_298M: 2753335 hashes fora do padrão (1.46% dos não-nulos)
+-------------------------------+
|collater_typofvalofguarant_298M|
+-------------------------------+
|8fd95e4b                       |
|9a0c095e                       |
|06fb9ba8                       |
|26cf31be                       |
+-------------------------------+

  ⚠️  collater_typofvalofguarant_407M: 6884250 hashes fora do padrão (3.66% dos não-nulos)
+-------------------------------+
|collater_typofvalofguarant_407M|
+-------------------------------+
|8fd95e4b                       |
|9a0c095e                       |
|06fb9ba8                       |
|3cbe86ba                       |
|9276e4bb                       |
+-------------------------------+
only showing top 5 rows

  ⚠️  collaterals_typeofguarante_359M: 6883826 hashes fora do padrão (3.66% dos não-nulos)
+-------------------------------+
|collaterals_typeofguarante_359M|
+----------

## 📋 Análise — `train_credit_bureau_a_2` (union de `_0` a `_10`)

## 📋 Análise — `train_credit_bureau_a_2` (union de `_0` a `_10`)

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| num_group1 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| num_group2 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| collater_typofvalofguarant_298M | string | manter | Código ofuscado, string correto |
| collater_typofvalofguarant_407M | string | manter | Código ofuscado, string correto |
| collater_valueofguarantee_1124L | double | manter | Valor de garantia, double correto |
| collater_valueofguarantee_876L | double | manter | Valor de garantia, double correto |
| collaterals_typeofguarante_359M | string | manter | Código ofuscado, string correto |
| collaterals_typeofguarante_669M | string | manter | Código ofuscado, string correto |
| pmts_dpd_1073P | double | manter | Métrica de atraso, double correto |
| pmts_dpd_303P | double | manter | Métrica de atraso, double correto |
| pmts_month_158T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| pmts_month_706T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| pmts_overdue_1140A | double | manter | Valor monetário em atraso, double correto |
| pmts_overdue_1152A | double | manter | Valor monetário em atraso, double correto |
| pmts_year_1139T | double | cast → IntegerType | Ano (2008–2021), double desnecessário |
| pmts_year_507T | double | cast → IntegerType | Ano (2000–2028), double desnecessário |
| subjectroles_name_541M | string | manter | Código ofuscado, string correto |
| subjectroles_name_838M | string | manter | Código ofuscado, string correto |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

### Nulos

| Classificação | Colunas | null_pct | Ação Silver |
|---|---|---|---|
| Estrutural | `collater_valueofguarantee_1124L`, `collater_valueofguarantee_876L` | 96–98% | Manter null — garantia ausente na maioria dos contratos |
| Informativo | `pmts_dpd_1073P`, `pmts_overdue_1140A` | ~81% | Manter null — pagamento sem atraso registrado |
| Informativo | `pmts_month_158T`, `pmts_year_1139T` | ~65% | Manter null — fonte com menor cobertura temporal |
| Informativo | `pmts_dpd_303P`, `pmts_overdue_1152A` | ~60% | Manter null — fonte alternativa com menor cobertura |
| Baixo | `pmts_month_706T`, `pmts_year_507T` | ~16% | Manter null |
| Zero | `case_id`, `num_group1`, `num_group2`, colunas `*M`, metadados | 0% | — |

⚠️ Após substituição de `a55475b1` por `null` nas colunas `*M`, o null_pct real será ~96–99% nessas colunas — alinhado com as colunas numéricas de garantia, confirmando que formam grupos funcionais.  
⚠️ Nenhuma imputação na Silver — todos os nulos preservados.

### Sentinelas

**Numéricos:**

| Coluna | Valores sentinela |
|---|---|
| `collater_valueofguarantee_876L` | 9999, 99999 |
| `pmts_dpd_1073P` | 99, 999 |
| `pmts_dpd_303P` | 99, 999, **-1** |
| `pmts_overdue_1140A` | 99, 999, 9999 |
| `pmts_overdue_1152A` | 99, 999, 9999 |

⚠️ `pmts_dpd_303P` com sentinela `-1` — segundo campo identificado no projeto com esse padrão (o primeiro foi `dpdmax_757P` em `train_credit_bureau_a_1`).

**Strings:** `a55475b1` detectado em todas as 6 colunas `*M` com volumes massivos (181M–185M ocorrências). Substituir por `null`.

### Cardinalidade
- `case_id`: ~1.34M distintos — mesma cobertura de `train_credit_bureau_a_1`, esperado (mesma fonte de bureau)
- `pmts_overdue_1152A`: ~3.3M distintos — valores monetários contínuos, esperado
- `num_group1`: 335 distintos — range menor que `train_credit_bureau_a_1` (518), reflete que `num_group2` absorve parte da granularidade
- `num_group2`: 104 distintos — representa pagamentos mensais por contrato; explica o volume de 188M de linhas
- `pmts_year_507T`: 27 anos (2000–2028) vs `pmts_year_1139T`: 13 anos (2008–2021) — fontes com coberturas temporais distintas; anos futuros em `pmts_year_507T` (2025–2028) são parcelas futuras planejadas, legítimos
- `pmts_month_*T`: cardinalidade 12 — meses confirmados, cast para `int` seguro
- Colunas `*M`: 5–14 categorias válidas após remoção de `a55475b1`

### Chave Composta
`(case_id, num_group1, num_group2)` é chave composta única (0 duplicatas em 188.298.452 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição
Tabela com **três níveis de granularidade**: `case_id` → `num_group1` (contrato) → `num_group2` (pagamento mensal do contrato).  
- `num_group1`: distribuição decrescente de 0 (42.9M) a 49+ — mesmo padrão de `train_credit_bureau_a_1`
- `num_group2`: até 104 distintos — representa até ~8 anos de histórico mensal por contrato (104 meses ≈ 8.7 anos)
- Volume total de 188M é o maior do projeto — resultado direto da granularidade mensal por contrato

### Colunas Codificadas (`*M`) — Dois Padrões de Codificação
Mesmo padrão identificado em `train_credit_bureau_a_1` — dois esquemas coexistentes:

| Coluna | Hashes fora do padrão | % dos não-nulos |
|---|---|---|
| `collater_typofvalofguarant_407M` | 6.884.250 | 3.66% |
| `collaterals_typeofguarante_359M` | 6.883.826 | 3.66% |
| `collaterals_typeofguarante_669M` | similar | ~3.66% |
| `collater_typofvalofguarant_298M` | 2.753.335 | 1.46% |
| `subjectroles_name_*M` | presente | < 1% |

**Ação Silver:** manter ambos os padrões como string. Substituir apenas `a55475b1` por `null`.

---

### Ações Silver — Casts

```python
int_cols = [
    "num_group1", "num_group2",
    "pmts_month_158T", "pmts_month_706T",
    "pmts_year_1139T", "pmts_year_507T"
]
for c in int_cols:
    df = df.withColumn(c, F.col(c).cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
sentinel_map = {
    "collater_valueofguarantee_876L": ,
    "pmts_dpd_1073P":                 ,
    "pmts_dpd_303P":                  [99, 999, -1],
    "pmts_overdue_1140A":             ,
    "pmts_overdue_1152A":             ,
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — Sentinelas em Colunas `*M` → null

```python
m_cols = [c for c in df.columns if c.endswith("M")]
for c in m_cols:
    df = df.withColumn(
        c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c))
    )
```

### Ações Silver — checks_silver.py

```python
# Unicidade da chave composta
dup_count = df.groupBy("case_id", "num_group1", "num_group2").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_credit_bureau_a_2: duplicatas na chave (case_id, num_group1, num_group2)"

# Auditoria de padrões em colunas *M
for c in [col for col in df.columns if col.endswith("M")]:
    total_notnull = df.filter(F.col(c).isNotNull()).count()
    pattern_p   = df.filter(F.col(c).rlike(r"^P\d+_\d+_\d+$")).count()
    pattern_hex = df.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull()
    ).count()
    print(f"{c}: padrão P={pattern_p} | hex={pattern_hex} | total={total_notnull}")
```

**Granularidade:** três níveis — `(case_id, num_group1, num_group2)` representando cliente → contrato → pagamento mensal.  
**Join com train_base:** `left join` por `case_id` — cobertura parcial esperada (~1.34M de 1.526M).  
**Volume:** 188M de linhas — maior tabela do projeto. Processar em chunks na Silver se necessário dado o ambiente `local[1]` com 3g de driver memory.  
**Anos futuros em `pmts_year_507T`** (2025–2028): parcelas futuras planejadas — legítimos, não tratar como anomalia.  
**Nulos:** todos preservados — imputação adiada para Gold.


## Dataset train_credit_bureau_b_1

In [22]:
df_credit_bureau_b_1 = load_logical_dataset(spark, bronze_dir, "train_credit_bureau_b_1")

Partições encontradas para 'train_credit_bureau_b_1':
  train_credit_bureau_b_1


In [23]:
df_credit_bureau_b_1.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- amount_1115A: double (nullable = true)
 |-- classificationofcontr_1114M: string (nullable = true)
 |-- contractdate_551D: string (nullable = true)
 |-- contractmaturitydate_151D: string (nullable = true)
 |-- contractst_516M: string (nullable = true)
 |-- contracttype_653M: string (nullable = true)
 |-- credlmt_1052A: double (nullable = true)
 |-- credlmt_228A: double (nullable = true)
 |-- credlmt_3940954A: double (nullable = true)
 |-- credor_3940957M: string (nullable = true)
 |-- credquantity_1099L: double (nullable = true)
 |-- credquantity_984L: double (nullable = true)
 |-- debtpastduevalue_732A: double (nullable = true)
 |-- debtvalue_227A: double (nullable = true)
 |-- dpd_550P: double (nullable = true)
 |-- dpd_733P: double (nullable = true)
 |-- dpdmax_851P: double (nullable = true)
 |-- dpdmaxdatemonth_804T: double (nullable = true)
 |-- dpdmaxdateyear_742T: double (nullable = true)
 |-- installmentamount_644A: double (nullable

In [24]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_credit_bureau_b_1.count()

null_counts = df_credit_bureau_b_1.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_credit_bureau_b_1.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_credit_bureau_b_1.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 85791

coluna                                          null_count   null_pct
----------------------------------------------------------------------
periodicityofpmts_997L                               83764     97.64%
interesteffectiverate_369L                           76285     88.92%
credlmt_228A                                         69661      81.2%
residualamount_1093A                                 69666      81.2%
credlmt_1052A                                        58210     67.85%
residualamount_127A                                  58210     67.85%
interestrateyearly_538L                              56966      66.4%
residualamount_3940956A                              48272     56.27%
credlmt_3940954A                                     47573     55.45%
instlamount_892A                                     42298      49.3%
numberofinstls_810L                                  42298      49.3%
pmtnumpending_403L                                   42111     49

In [25]:
# ── SENTINELAS ───────────────────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, 999999, -1]

numeric_cols = [c for c, t in df_credit_bureau_b_1.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1")]

for c in numeric_cols:
    counts = df_credit_bureau_b_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")

print("\n=== SENTINELAS — Strings ===")

sentinel_strs = ["XNA", "Unknown", "unknown", "N/A", "na", "", "a55475b1"]

string_cols = [c for c, t in df_credit_bureau_b_1.dtypes
               if t == "string"
               and c not in ("_run_id", "_source_file")]

for c in string_cols:
    counts = df_credit_bureau_b_1.select([
        F.sum((F.col(c) == v).cast("int")).alias(v if v else "empty")
        for v in sentinel_strs
    ]).first()
    hits = {v: counts[v if v else "empty"] for v in sentinel_strs
            if counts[v if v else "empty"] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
  dpd_550P: {99: 1}
  dpdmax_851P: {99: 6, 9999: 2}
  installmentamount_833A: {9999: 2}
  instlamount_892A: {999: 1}
  maxdebtpduevalodued_3940955A: {99: 3}
  numberofinstls_810L: {99: 8}
  overdueamountmax_950A: {99: 2}
  pmtdaysoverdue_1135P: {99: 6}
  pmtnumpending_403L: {99: 13}
  residualamount_127A: {9999: 2}
  residualamount_3940956A: {9999: 4}

=== SENTINELAS — Strings ===
  classificationofcontr_1114M: {'a55475b1': 4060}
  contractdate_551D: nenhum sentinela detectado
  contractmaturitydate_151D: nenhum sentinela detectado
  contractst_516M: {'a55475b1': 4121}
  contracttype_653M: {'a55475b1': 3892}
  credor_3940957M: {'a55475b1': 3892}
  lastupdate_260D: nenhum sentinela detectado
  periodicityofpmts_997L: nenhum sentinela detectado
  periodicityofpmts_997M: {'a55475b1': 41057}
  pmtmethod_731M: {'a55475b1': 38483}
  purposeofcred_722M: {'a55475b1': 3892}
  subjectrole_326M: {'a55475b1': 32773}
  subjectrole_43M: {'a55475b1': 39563}


In [26]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_credit_bureau_b_1.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_credit_bureau_b_1.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    flag = " ← constante" if nunique == 1 else ""
    print(f"{coluna:<45} {nunique:>15}{flag}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
debtvalue_227A                                          40290
installmentamount_833A                                  39701
case_id                                                 38099
instlamount_892A                                        32457
totalamount_881A                                        28853
totalamount_503A                                        25087
residualamount_3940956A                                 23058
amount_1115A                                            20454
dpdmax_851P                                             19744
residualamount_127A                                     17182
credlmt_1052A                                            9811
credlmt_3940954A                                         9656
debtpastduevalue_732A                                    5770
dpd_550P                            

In [27]:
# ── CHAVE COMPOSTA ───────────────────────────────────────────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===")

unique_keys = df_credit_bureau_b_1.select("case_id", "num_group1").distinct().count()
dups        = total - unique_keys

print(f"  Total de linhas : {total}")
print(f"  Chaves únicas   : {unique_keys}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_credit_bureau_b_1.groupBy("case_id", "num_group1").count() \
                        .filter(F.col("count") > 1) \
                        .orderBy(F.col("count").desc()) \
                        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_credit_bureau_b_1.groupBy("num_group1").count() \
                    .orderBy("num_group1") \
                    .show(50, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1) ===
  Total de linhas : 85791
  Chaves únicas   : 85791
  Duplicatas      : 0
  ✅ (case_id, num_group1) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+-----+
|num_group1|count|
+----------+-----+
|0         |36500|
|1         |27670|
|2         |12365|
|3         |5506 |
|4         |2185 |
|5         |894  |
|6         |370  |
|7         |163  |
|8         |67   |
|9         |32   |
|10        |11   |
|11        |8    |
|12        |6    |
|13        |4    |
|14        |2    |
|15        |2    |
|16        |2    |
|17        |1    |
|18        |1    |
|19        |1    |
|20        |1    |
+----------+-----+



In [28]:
# ── RANGE DE DATAS ───────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

date_cols = [c for c, t in df_credit_bureau_b_1.dtypes
             if t == "string" and c.endswith("D")
             and c not in ("_run_id", "_source_file")]

df_credit_bureau_b_1.select([
    val
    for c in date_cols
    for val in (F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max"))
]).show(truncate=False)


=== RANGE DE DATAS ===
+---------------------+---------------------+-----------------------------+-----------------------------+-------------------+-------------------+
|contractdate_551D_min|contractdate_551D_max|contractmaturitydate_151D_min|contractmaturitydate_151D_max|lastupdate_260D_min|lastupdate_260D_max|
+---------------------+---------------------+-----------------------------+-----------------------------+-------------------+-------------------+
|2000-01-15           |2020-10-14           |2003-04-11                   |2045-08-17                   |2015-03-30         |2020-10-19         |
+---------------------+---------------------+-----------------------------+-----------------------------+-------------------+-------------------+



In [29]:
# ── COLUNAS *M — PADRÃO E HASHES ─────────────────────────────────────────────
print("=== VALORES FORA DO PADRÃO — Colunas *M ===")

code_cols = [c for c in df_credit_bureau_b_1.columns if c.endswith("M")]

for c in code_cols:
    total_notnull = df_credit_bureau_b_1.filter(F.col(c).isNotNull()).count()
    outliers = df_credit_bureau_b_1.filter(
        ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
        F.col(c).isNotNull() &
        (F.col(c) != "a55475b1")
    ).count()
    pct = round(outliers / total_notnull * 100, 2) if total_notnull > 0 else 0
    if outliers > 0:
        print(f"  ⚠️  {c}: {outliers} hashes ({pct}% dos não-nulos)")
        df_credit_bureau_b_1.filter(
            ~F.col(c).rlike(r"^P\d+_\d+_\d+$") &
            F.col(c).isNotNull() &
            (F.col(c) != "a55475b1")
        ).select(c).distinct().show(5, truncate=False)
    else:
        print(f"  ✅ {c}: todos no padrão esperado")


=== VALORES FORA DO PADRÃO — Colunas *M ===
  ⚠️  classificationofcontr_1114M: 81731 hashes (95.27% dos não-nulos)
+---------------------------+
|classificationofcontr_1114M|
+---------------------------+
|ea6782cc                   |
|436d55c2                   |
|1cf4e481                   |
|90c587b1                   |
|00135d9c                   |
+---------------------------+
only showing top 5 rows

  ⚠️  contractst_516M: 81670 hashes (95.2% dos não-nulos)
+---------------+
|contractst_516M|
+---------------+
|83931972       |
|dd67cff0       |
|54132f86       |
|04bf6e27       |
|7640edc3       |
+---------------+
only showing top 5 rows

  ⚠️  contracttype_653M: 81899 hashes (95.46% dos não-nulos)
+-----------------+
|contracttype_653M|
+-----------------+
|1c9c5356         |
|f4e17141         |
|60e784d6         |
|190917cc         |
|920c2ccd         |
+-----------------+
only showing top 5 rows

  ⚠️  credor_3940957M: 52847 hashes (61.6% dos não-nulos)
+---------------+
|cr

In [30]:
# ── PARES SIMILARES ──────────────────────────────────────────────────────────
print("=== SOBREPOSIÇÃO DE PARES SIMILARES ===")

pairs = [
    ("credlmt_1052A",         "credlmt_228A"),
    ("credlmt_228A",          "credlmt_3940954A"),
    ("dpd_550P",              "dpd_733P"),
    ("installmentamount_644A","installmentamount_833A"),
    ("residualamount_1093A",  "residualamount_127A"),
    ("totalamount_503A",      "totalamount_881A"),
    ("credquantity_1099L",    "credquantity_984L"),
]

print(f"{'Par':<55} {'ambos':>8} {'só_A':>10} {'só_B':>10}")
print("-" * 85)

for a, b in pairs:
    both   = df_credit_bureau_b_1.filter(F.col(a).isNotNull() & F.col(b).isNotNull()).count()
    only_a = df_credit_bureau_b_1.filter(F.col(a).isNotNull() & F.col(b).isNull()).count()
    only_b = df_credit_bureau_b_1.filter(F.col(a).isNull()    & F.col(b).isNotNull()).count()
    print(f"{a+' / '+b:<55} {both:>8} {only_a:>10} {only_b:>10}")


=== SOBREPOSIÇÃO DE PARES SIMILARES ===
Par                                                        ambos       só_A       só_B
-------------------------------------------------------------------------------------
credlmt_1052A / credlmt_228A                               11922      15659       4208
credlmt_228A / credlmt_3940954A                            11922       4208      26296
dpd_550P / dpd_733P                                        39290      13728       6938
installmentamount_644A / installmentamount_833A            39290       6938      13728
residualamount_1093A / residualamount_127A                 11922       4203      15659
totalamount_503A / totalamount_881A                        39290      13728       6938
credquantity_1099L / credquantity_984L                     39290      13728       6938


## 📋 Análise — `train_credit_bureau_b_1`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| num_group1 | long | cast → IntegerType | Range 0–20, long desnecessário |
| amount_1115A | double | manter | Valor monetário, double correto |
| classificationofcontr_1114M | string | manter | Código ofuscado, string correto |
| contractdate_551D | string | cast → DateType | Data armazenada como string |
| contractmaturitydate_151D | string | cast → DateType | Data armazenada como string |
| contractst_516M | string | manter | Código ofuscado, string correto |
| contracttype_653M | string | manter | Código ofuscado, string correto |
| credlmt_1052A | double | manter | Limite de crédito, double correto |
| credlmt_228A | double | manter | Limite de crédito, double correto |
| credlmt_3940954A | double | manter | Limite de crédito, double correto |
| credor_3940957M | string | manter | Código ofuscado, string correto |
| credquantity_1099L | double | cast → IntegerType | Contagem (14 distintos), double desnecessário |
| credquantity_984L | double | cast → IntegerType | Contagem (70 distintos), double desnecessário |
| debtpastduevalue_732A | double | manter | Valor monetário, double correto |
| debtvalue_227A | double | manter | Valor monetário, double correto |
| dpd_550P | double | manter | Métrica de atraso, double correto |
| dpd_733P | double | manter | Métrica de atraso, double correto |
| dpdmax_851P | double | manter | Métrica de atraso, double correto |
| dpdmaxdatemonth_804T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| dpdmaxdateyear_742T | double | cast → IntegerType | Ano (15 distintos), double desnecessário |
| installmentamount_644A | double | manter | Valor monetário, double correto |
| installmentamount_833A | double | manter | Valor monetário, double correto |
| instlamount_892A | double | manter | Valor monetário, double correto |
| interesteffectiverate_369L | double | manter | Taxa percentual, double correto |
| interestrateyearly_538L | double | manter | Taxa percentual, double correto |
| lastupdate_260D | string | cast → DateType | Data armazenada como string |
| maxdebtpduevalodued_3940955A | double | manter | Valor monetário, double correto |
| numberofinstls_810L | double | cast → IntegerType | Contagem de parcelas, double desnecessário |
| overdueamountmax_950A | double | manter | Valor monetário, double correto |
| overdueamountmaxdatemonth_494T | double | cast → IntegerType | Mês (1–12), double desnecessário |
| overdueamountmaxdateyear_432T | double | cast → IntegerType | Ano (15 distintos), double desnecessário |
| periodicityofpmts_997L | double | cast → IntegerType | 5 valores distintos — periodicidade, double desnecessário |
| periodicityofpmts_997M | string | manter | Mesma semântica que `997L` mas como string — ver nota abaixo |
| pmtdaysoverdue_1135P | double | manter | Dias em atraso, double correto |
| pmtmethod_731M | string | manter | Código ofuscado, string correto |
| pmtnumpending_403L | double | cast → IntegerType | Contagem de parcelas pendentes, double desnecessário |
| purposeofcred_722M | string | manter | Código ofuscado, string correto |
| residualamount_1093A | double | manter | Valor monetário, double correto |
| residualamount_127A | double | manter | Valor monetário, double correto |
| residualamount_3940956A | double | manter | Valor monetário, double correto |
| subjectrole_326M | string | manter | Código ofuscado, string correto |
| subjectrole_43M | string | manter | Código ofuscado, string correto |
| totalamount_503A | double | manter | Valor monetário, double correto |
| totalamount_881A | double | manter | Valor monetário, double correto |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

⚠️ **`periodicityofpmts_997L` e `periodicityofpmts_997M`**: mesma semântica com sufixos diferentes (`L` = numérico, `M` = código ofuscado). São fontes distintas para o mesmo conceito — manter ambas na Silver. `997L` tem apenas 5 valores e 97.64% de nulos; `997M` tem 9 valores e 3.81% de nulos — cobertura muito diferente, não são redundantes.

⚠️ **`residualamount_1093A`**: apenas 2 valores distintos com 81.2% de nulos — investigar na Gold se é flag binário disfarçado de valor monetário.

### Nulos

| Classificação | null_pct | Colunas representativas |
|---|---|---|
| Estrutural alto | ~97% | `periodicityofpmts_997L` |
| Estrutural alto | ~81–89% | `credlmt_228A`, `residualamount_1093A`, `interesteffectiverate_369L` |
| Informativo | ~55–68% | `credlmt_1052A`, `residualamount_127A`, `interestrateyearly_538L`, `credlmt_3940954A` |
| Grupos funcionais | ~49% | `instlamount_892A`, `numberofinstls_810L`, `amount_1115A`, `debtvalue_227A` (null_pct idêntico) |
| Grupos funcionais | ~38–46% | `credquantity_984L`, `dpd_733P`, `installmentamount_644A`, `totalamount_881A` / `credquantity_1099L`, `dpd_550P`, `installmentamount_833A`, `totalamount_503A` |
| Baixo | ~5% | `debtpastduevalue_732A`, `dpdmax_851P`, `pmtdaysoverdue_1135P` e adjacentes |
| Residual | ~4% | `contractmaturitydate_151D`, `contractdate_551D`, `lastupdate_260D` |
| Zero | 0% | `case_id`, `num_group1`, colunas `*M` base, metadados |

⚠️ Nenhuma imputação na Silver — todos os nulos preservados.

### Sentinelas

**Numéricos:**

| Coluna | Valores sentinela |
|---|---|
| `dpd_550P` | 99 |
| `dpdmax_851P` | 99, 9999 |
| `installmentamount_833A` | 9999 |
| `instlamount_892A` | 999 |
| `maxdebtpduevalodued_3940955A` | 99 |
| `numberofinstls_810L` | 99 |
| `overdueamountmax_950A` | 99 |
| `pmtdaysoverdue_1135P` | 99 |
| `pmtnumpending_403L` | 99 |
| `residualamount_127A` | 9999 |
| `residualamount_3940956A` | 9999 |

**Strings:** `a55475b1` em 9 colunas `*M`. Substituir por `null`.

### Cardinalidade
- `case_id`: ~38.099 distintos — cobertura muito menor que demais tabelas de bureau (~1.34M); essa é uma fonte de bureau menor/alternativa
- `residualamount_1093A`: apenas 2 distintos com 81.2% de nulos — investigar na Gold
- `installmentamount_644A`: apenas 85 distintos para valor monetário — possível valor discretizado ou padrão
- `num_group1`: 21 distintos (0–20) — distribuição fortemente concentrada em 0 e 1

### Chave Composta
`(case_id, num_group1)` é chave composta única (0 duplicatas em 85.791 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição de `num_group1`
Distribuição fortemente decrescente: grupo 0 com 36.500 linhas vs grupo 20 com apenas 1. A maioria dos clientes tem 1–2 contratos nessa fonte de bureau.

### Range de Datas
Todos os ranges dentro de domínio plausível:
- `contractdate_551D`: 2000-01-15 a 2020-10-14 ✅
- `contractmaturitydate_151D`: 2003-04-11 a 2045-08-17 ✅ (datas futuras legítimas — contratos de longo prazo)
- `lastupdate_260D`: 2015-03-30 a 2020-10-19 ✅

Nenhum sentinela de data detectado. Casts seguros.

### Colunas `*M` — Dois Padrões de Codificação
A maioria das colunas `*M` tem hashes como **padrão dominante** nessa tabela (95%+ dos não-nulos), invertendo o padrão das tabelas anteriores onde `P\d+_\d+_\d+` era majoritário. Confirma que `train_credit_bureau_b` é uma fonte de bureau diferente com esquema de codificação distinto.

| Coluna | Hashes (% dos não-nulos) |
|---|---|
| `classificationofcontr_1114M` | 95.27% |
| `contractst_516M` | 95.20% |
| demais `*M` | presente em proporções variadas |

**Ação Silver:** manter ambos os padrões como string. Substituir apenas `a55475b1` por `null`.

### Pares de Colunas Similares
Todos os pares têm sobreposição relevante — não são mutuamente exclusivos:

| Par | Ambos | Só A | Só B | Estratégia Gold |
|---|---|---|---|---|
| `dpd_550P` / `dpd_733P` | 39.290 | 13.728 | 6.938 | Features independentes |
| `installmentamount_644A` / `installmentamount_833A` | 39.290 | 6.938 | 13.728 | Features independentes |
| `totalamount_503A` / `totalamount_881A` | 39.290 | 13.728 | 6.938 | Features independentes |
| `credquantity_1099L` / `credquantity_984L` | 39.290 | 13.728 | 6.938 | Features independentes |
| `credlmt_1052A` / `credlmt_228A` | 11.922 | 15.659 | 4.208 | Avaliar `coalesce` |
| `credlmt_228A` / `credlmt_3940954A` | 11.922 | 4.208 | 26.296 | Avaliar `coalesce` |
| `residualamount_1093A` / `residualamount_127A` | 11.922 | 4.203 | 15.659 | Avaliar `coalesce` |

**Ação Silver:** manter todos os pares intactos. Consolidação adiada para Gold.

---

### Ações Silver — Casts

```python
date_cols = ["contractdate_551D", "contractmaturitydate_151D", "lastupdate_260D"]
for c in date_cols:
    df = df.withColumn(c, F.to_date(F.col(c), "yyyy-MM-dd"))

int_cols = [
    "num_group1", "credquantity_1099L", "credquantity_984L",
    "dpdmaxdatemonth_804T", "dpdmaxdateyear_742T",
    "numberofinstls_810L", "overdueamountmaxdatemonth_494T",
    "overdueamountmaxdateyear_432T", "periodicityofpmts_997L",
    "pmtnumpending_403L"
]
for c in int_cols:
    df = df.withColumn(c, F.col(c).cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
sentinel_map = {
    "dpd_550P":                     ,
    "dpdmax_851P":                  ,
    "installmentamount_833A":       ,
    "instlamount_892A":             ,
    "maxdebtpduevalodued_3940955A": ,
    "numberofinstls_810L":          ,
    "overdueamountmax_950A":        ,
    "pmtdaysoverdue_1135P":         ,
    "pmtnumpending_403L":           ,
    "residualamount_127A":          ,
    "residualamount_3940956A":      ,
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — Sentinelas em Colunas `*M` → null

```python
m_cols = [c for c in df.columns if c.endswith("M")]
for c in m_cols:
    df = df.withColumn(
        c, F.when(F.col(c) == "a55475b1", None).otherwise(F.col(c))
    )
```

### Ações Silver — checks_silver.py

```python
dup_count = df.groupBy("case_id", "num_group1").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_credit_bureau_b_1: duplicatas na chave (case_id, num_group1)"

# Investigar residualamount_1093A — apenas 2 valores distintos
df.filter(F.col("residualamount_1093A").isNotNull()) \
  .select("residualamount_1093A").distinct().show(truncate=False)
```

**Granularidade:** N linhas por `case_id` (máx. 20) — chave composta `(case_id, num_group1)`.  
**Cobertura:** ~38K `case_id` distintos — fonte de bureau menor, cobertura parcial do `train_base`.  
**Join com train_base:** `left join` por `case_id`.  
**`periodicityofpmts_997L` vs `997M`:** fontes distintas para o mesmo conceito — coberturas muito diferentes (97.64% vs 3.81% de nulos), manter ambas.  
**Nulos:** todos preservados — imputação adiada para Gold.


## Dataset train_credit_bureau_b_2

In [31]:
df_credit_bureau_b_2 = load_logical_dataset(spark, bronze_dir, "train_credit_bureau_b_2")

Partições encontradas para 'train_credit_bureau_b_2':
  train_credit_bureau_b_2


In [32]:
df_credit_bureau_b_2.printSchema()

root
 |-- case_id: long (nullable = true)
 |-- num_group1: long (nullable = true)
 |-- num_group2: long (nullable = true)
 |-- pmts_date_1107D: string (nullable = true)
 |-- pmts_dpdvalue_108P: double (nullable = true)
 |-- pmts_pmtsoverdue_635A: double (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _ingest_date: date (nullable = true)
 |-- _source_file: string (nullable = true)



In [33]:
from __future__ import annotations

from pyspark.sql import functions as F

# ── NULOS ────────────────────────────────────────────────────────────────────
total = df_credit_bureau_b_2.count()

null_counts = df_credit_bureau_b_2.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_credit_bureau_b_2.columns
]).first()

rows = sorted(
    [(c, int(null_counts[c]), round(int(null_counts[c]) / total * 100, 2))
     for c in df_credit_bureau_b_2.columns],
    key=lambda x: x[2], reverse=True
)

print(f"Total de linhas: {total}\n")
print(f"{'coluna':<45} {'null_count':>12} {'null_pct':>10}")
print("-" * 70)
for coluna, null_count, null_pct in rows:
    print(f"{coluna:<45} {null_count:>12} {null_pct:>9}%")


Total de linhas: 1286755

coluna                                          null_count   null_pct
----------------------------------------------------------------------
pmts_dpdvalue_108P                                    5361      0.42%
pmts_pmtsoverdue_635A                                 5361      0.42%
case_id                                                  0       0.0%
num_group1                                               0       0.0%
num_group2                                               0       0.0%
pmts_date_1107D                                          0       0.0%
_run_id                                                  0       0.0%
_ingest_date                                             0       0.0%
_source_file                                             0       0.0%


In [34]:
# ── SENTINELAS ───────────────────────────────────────────────────────────────
print("=== SENTINELAS — Numéricos ===")

sentinel_vals = [99, 999, 9999, 99999, -1]

numeric_cols = [c for c, t in df_credit_bureau_b_2.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("case_id", "num_group1", "num_group2")]

for c in numeric_cols:
    counts = df_credit_bureau_b_2.select([
        F.sum((F.col(c) == v).cast("int")).alias(str(v))
        for v in sentinel_vals
    ]).first()
    hits = {v: counts[str(v)] for v in sentinel_vals if counts[str(v)] > 0}
    if hits:
        print(f"  {c}: {hits}")
    else:
        print(f"  {c}: nenhum sentinela detectado")


=== SENTINELAS — Numéricos ===
  pmts_dpdvalue_108P: {99: 14, 999: 2, 9999: 5, 99999: 1}
  pmts_pmtsoverdue_635A: {99: 33}


In [35]:
# ── CARDINALIDADE ────────────────────────────────────────────────────────────
print("=== CARDINALIDADE POR COLUNA ===")

card_cols = [c for c in df_credit_bureau_b_2.columns
             if c not in ("_run_id", "_source_file", "_ingest_date")]

card_counts = df_credit_bureau_b_2.select([
    F.approx_count_distinct(F.col(c)).alias(c)
    for c in card_cols
]).first()

rows_card = sorted(
    [(c, int(card_counts[c])) for c in card_cols],
    key=lambda x: x[1], reverse=True
)

print(f"{'coluna':<45} {'approx_nunique':>15}")
print("-" * 62)
for coluna, nunique in rows_card:
    print(f"{coluna:<45} {nunique:>15}")


=== CARDINALIDADE POR COLUNA ===
coluna                                         approx_nunique
--------------------------------------------------------------
pmts_dpdvalue_108P                                      59690
case_id                                                 37987
pmts_pmtsoverdue_635A                                    3921
pmts_date_1107D                                            57
num_group2                                                 37
num_group1                                                 21


In [36]:
# ── CHAVE COMPOSTA ───────────────────────────────────────────────────────────
print("=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===")

unique_keys = df_credit_bureau_b_2.select(
    "case_id", "num_group1", "num_group2"
).distinct().count()
dups = total - unique_keys

print(f"  Total de linhas : {total}")
print(f"  Chaves únicas   : {unique_keys}")
print(f"  Duplicatas      : {dups}")

if dups == 0:
    print("  ✅ (case_id, num_group1, num_group2) é chave composta única")
else:
    print("  ⚠️  Duplicatas encontradas — investigar")
    df_credit_bureau_b_2.groupBy("case_id", "num_group1", "num_group2").count() \
                        .filter(F.col("count") > 1) \
                        .orderBy(F.col("count").desc()) \
                        .show(10, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group1 ===")
df_credit_bureau_b_2.groupBy("num_group1").count() \
                    .orderBy("num_group1").show(30, truncate=False)

print("\n=== DISTRIBUIÇÃO DE num_group2 ===")
df_credit_bureau_b_2.groupBy("num_group2").count() \
                    .orderBy("num_group2").show(30, truncate=False)


=== UNICIDADE DA CHAVE COMPOSTA — (case_id, num_group1, num_group2) ===
  Total de linhas : 1286755
  Chaves únicas   : 1286755
  Duplicatas      : 0
  ✅ (case_id, num_group1, num_group2) é chave composta única

=== DISTRIBUIÇÃO DE num_group1 ===
+----------+------+
|num_group1|count |
+----------+------+
|0         |723766|
|1         |327250|
|2         |139937|
|3         |58034 |
|4         |22693 |
|5         |8694  |
|6         |3354  |
|7         |1414  |
|8         |780   |
|9         |349   |
|10        |169   |
|11        |101   |
|12        |59    |
|13        |20    |
|14        |23    |
|15        |39    |
|16        |16    |
|17        |5     |
|18        |10    |
|19        |36    |
|20        |6     |
+----------+------+


=== DISTRIBUIÇÃO DE num_group2 ===
+----------+-----+
|num_group2|count|
+----------+-----+
|0         |81889|
|1         |78332|
|2         |73867|
|3         |69081|
|4         |64582|
|5         |60414|
|6         |56217|
|7         |52446|
|8     

In [37]:
# ── RANGE DE DATAS ───────────────────────────────────────────────────────────
print("=== RANGE DE DATAS ===")

df_credit_bureau_b_2.select(
    F.min("pmts_date_1107D").alias("pmts_date_min"),
    F.max("pmts_date_1107D").alias("pmts_date_max")
).show(truncate=False)

# Verificar sentinelas de data conhecidos
print("\n=== SENTINELAS DE DATA ===")
sentinel_dates = ["1900-01-15", "2098-01-15"]
for sd in sentinel_dates:
    count = df_credit_bureau_b_2.filter(F.col("pmts_date_1107D") == sd).count()
    print(f"  {sd}: {count} ocorrências")


=== RANGE DE DATAS ===
+-------------+-------------+
|pmts_date_min|pmts_date_max|
+-------------+-------------+
|2016-01-15   |2020-10-15   |
+-------------+-------------+


=== SENTINELAS DE DATA ===
  1900-01-15: 0 ocorrências
  2098-01-15: 0 ocorrências


## 📋 Análise — `train_credit_bureau_b_2`

### Schema e Tipos

| Coluna | dtype Bronze | Ação Silver | Justificativa |
|---|---|---|---|
| case_id | long | manter | Parte da chave composta, tipo correto |
| num_group1 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| num_group2 | long | cast → IntegerType | Índice ordinal, long desnecessário |
| pmts_date_1107D | string | cast → DateType | Data armazenada como string |
| pmts_dpdvalue_108P | double | manter | Métrica de atraso, double correto |
| pmts_pmtsoverdue_635A | double | manter | Valor monetário em atraso, double correto |
| _run_id | string | propagar | Metadado Bronze, não alterar |
| _ingest_date | date | propagar | Metadado Bronze, não alterar |
| _source_file | string | propagar | Metadado Bronze, não alterar |

### Nulos
Tabela praticamente completa — apenas `pmts_dpdvalue_108P` e `pmts_pmtsoverdue_635A` com 0.42% de nulos cada (5.361 linhas), com null_pct idêntico confirmando que formam um grupo funcional (quando um é nulo, o outro também é). Nenhuma ação necessária.

### Sentinelas

| Coluna | Valores sentinela |
|---|---|
| `pmts_dpdvalue_108P` | 99, 999, 9999, 99999 |
| `pmts_pmtsoverdue_635A` | 99 |

### Cardinalidade
- `case_id`: ~37.987 distintos — mesma cobertura de `train_credit_bureau_b_1` (~38K), confirmando que ambas são da mesma fonte de bureau
- `pmts_date_1107D`: 57 distintos — ~4.75 anos mensais (2016–2020), consistente com o range
- `num_group2`: 37 distintos — até 37 pagamentos mensais por contrato (~3 anos de histórico)
- `num_group1`: 21 distintos (0–20) — alinhado com `train_credit_bureau_b_1`

### Chave Composta
`(case_id, num_group1, num_group2)` é chave composta única (0 duplicatas em 1.286.755 linhas). Nenhuma deduplicação necessária na Silver.

### Granularidade e Distribuição
Três níveis: `case_id` → `num_group1` (contrato) → `num_group2` (pagamento mensal).  
`num_group1` fortemente concentrado em 0 (723.766) com queda acentuada — mesmo padrão de `train_credit_bureau_b_1`.  
`num_group2` vai até 37 — histórico de até ~3 anos mensais por contrato, menor que `train_credit_bureau_a_2` (104 meses).

### Range de Datas
`pmts_date_1107D`: 2016-01-15 a 2020-10-15 — range válido e sem sentinelas. Cast seguro.

---

### Ações Silver — Casts

```python
df = df.withColumn("pmts_date_1107D", F.to_date(F.col("pmts_date_1107D"), "yyyy-MM-dd"))
df = df.withColumn("num_group1", F.col("num_group1").cast("int"))
df = df.withColumn("num_group2", F.col("num_group2").cast("int"))
```

### Ações Silver — Sentinelas Numéricos → null

```python
sentinel_map = {
    "pmts_dpdvalue_108P":    ,
    "pmts_pmtsoverdue_635A": ,
}

for col, sentinels in sentinel_map.items():
    df = df.withColumn(
        col, F.when(F.col(col).isin(sentinels), None).otherwise(F.col(col))
    )
```

### Ações Silver — checks_silver.py

```python
dup_count = df.groupBy("case_id", "num_group1", "num_group2").count() \
              .filter(F.col("count") > 1).count()
assert dup_count == 0, \
    "train_credit_bureau_b_2: duplicatas na chave (case_id, num_group1, num_group2)"
```

**Granularidade:** três níveis — `(case_id, num_group1, num_group2)`.  
**Cobertura:** ~38K `case_id` — mesma fonte de `train_credit_bureau_b_1`.  
**Join com train_base:** `left join` por `case_id`.  
**Nulos:** praticamente inexistentes — tabela mais limpa do projeto até agora.
